# IEEE-CIS Fraud Detection: Feature Engineering (Final)

Break Through Tech AI Studio: Mastercard project, Banking 1B.

This is the team's final feature engineering notebook. It combines Eric's notebook (time split, category handling, email and device groups, frequency encoding, account history, amount and weekday features, missingness counts), Jean's (device rules from the EDA, missingness profile clusters), Salvador's (Kaggle-discussion insights: treating the `id_14` timezone as a category and flagging three-decimal amounts), Jennifer's (target encoding, new-device/email/address flags, velocity, and account amount statistics), and Susie's (7-day and 30-day velocity windows), and adds the pieces the challenge advisor (Debasmita) asked for and the gaps the EDA left open. Sections taken from a teammate's notebook say so in their headings or text, and Section 15 lists where each teammate's work ended up.

**Starting data.** The raw Kaggle files, loaded and joined exactly as in the EDA notebook. The training period is the same 472,432 transactions the EDA analyzed.

| Advisor advice or EDA gap | Where it is handled |
|---|---|
| Avoid one-hot encoding for categories with many values | Section 4 (routing by cardinality) and Section 6 (frequency encoding) |
| Explore mean encoding; it only works if fraud rates per category are stable over time | Section 10b (Jennifer): smoothed, out-of-time target encoding with a stability table; `all_no_jennifer` allows a with/without test |
| Trees need heavy derived features; neural nets need normalization | Two outputs: a tree table (Section 10) and a normalized neural-net table (Section 11) |
| Start with all features, then reduce without losing more than 1 to 2% | Named feature sets, including a reduced-V set (Sections 9 and 10) |
| Track detection rate and false positives with a KS table | Section 13 (reusable function, ready for the modeling pipeline) |
| Keep a config per run | `CONFIG` is saved next to every output (Section 14) |
| EDA gap: address-missing transactions are 11.4% of transactions but 37% of fraud, and get no account history | Section 7 (second account key built from card + email) |
| EDA decision: one missing indicator per missingness block | Section 8 |

**Split.** This notebook uses Eric's 80/10/10 time split. Its training period is the same 472,432 transactions the EDA used; the EDA's 20% validation period is divided into a validation half and a labeled final holdout, which is the out-of-time set the advisor asked for. A check in Section 2 confirms the training period matches the EDA.

**Nothing here trains a model.** Every rule (category vocabularies, frequency encodings, medians, scalers, V selection) is learned from the training period only and then applied unchanged to validation, holdout, and Kaggle test.

Run top to bottom (**Restart and Run All**). Expect several GB of RAM on the full data.

In [1]:
# Run once, then restart the kernel
# pip install kagglehub pandas numpy pyarrow scikit-learn matplotlib


## 1. Configuration

Every setting that can change an output lives here. The whole dictionary is saved with the outputs, so each run can be compared with the next one (the advisor's one-config-per-run advice). Change `run_name` whenever you change a setting.

In [2]:
import os
import gc
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import warnings
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

CONFIG = {
    "run_name": "fe_final_v1",
    "data_path": None,                  # folder with the Kaggle CSVs; None downloads with kagglehub
    "output_dir": "outputs/feature_engineering",
    "train_share": 0.80,                # same training period as the EDA
    "validation_share": 0.10,           # next 10% validation, last 10% labeled holdout
    "expected_train_rows": 472_432,     # from the EDA output; set to None for a smoke run
    "seed": 42,

    # Categorical routing (advisor: one-hot only when a column has few categories)
    "onehot_max_levels": 10,
    "encode_candidates": [
        "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6", "addr1", "addr2",
        "P_emaildomain", "R_emaildomain", "P_provider", "device_family", "DeviceType",
        "id_14", "id_30", "id_31", "id_33", "M4",   # id_14: timezone offset (from Salvador)
    ],


    # V-column reduction (EDA method)
    "v_corr_threshold": 0.95,
    "v_sample_rows": 100_000,

    "save_nn_track": True,
}

SECONDS_PER_DAY = 86_400
MISSING = "__MISSING__"
UNSEEN = "__UNSEEN__"
OUT = Path(CONFIG["output_dir"]) / CONFIG["run_name"]

## 2. Load, join, and split by time

Loading follows the EDA exactly: left-join identity onto transactions, rename test's `id-01` columns, downcast floats to `float32` (keeping `TransactionAmt` as `float64`). The split follows Eric's notebook: sort by time, then move each boundary to the start of a timestamp so payments made at the same second never land in two periods.

In [3]:
def downcast(df, skip=("TransactionAmt",)):
    floats = [c for c in df.select_dtypes("float64").columns if c not in skip]
    df[floats] = df[floats].astype("float32")
    return df


def load(path, prefix):
    tx = pd.read_csv(os.path.join(path, f"{prefix}_transaction.csv"))
    ident = pd.read_csv(os.path.join(path, f"{prefix}_identity.csv"))
    ident.columns = ident.columns.str.replace("-", "_")
    df = tx.merge(ident, on="TransactionID", how="left", validate="one_to_one")
    assert len(df) == len(tx), "join changed the number of payments"
    return downcast(df)


path = CONFIG["data_path"]
if path is None:
    import kagglehub
    kagglehub.login()
    path = kagglehub.competition_download("ieee-fraud-detection")

labeled = load(path, "train")
kaggle_test = load(path, "test")

# Original inputs, recorded before anything is added
raw_cols = [c for c in labeled.columns if c not in ("TransactionID", "TransactionDT", "isFraud")]

labeled = labeled.sort_values(["TransactionDT", "TransactionID"]).reset_index(drop=True)
kaggle_test = kaggle_test.sort_values(["TransactionDT", "TransactionID"]).reset_index(drop=True)
times = labeled["TransactionDT"].to_numpy()

def boundary(fraction):
    row = min(int(len(times) * fraction), len(times) - 1)
    return int(np.searchsorted(times, times[row], side="left"))

train_end = boundary(CONFIG["train_share"])
val_end = boundary(CONFIG["train_share"] + CONFIG["validation_share"])
raw = {
    "train": labeled.iloc[:train_end].copy(),
    "validation": labeled.iloc[train_end:val_end].copy(),
    "holdout": labeled.iloc[val_end:].copy(),
    "kaggle_test": kaggle_test,
}
del labeled, kaggle_test, times
gc.collect()

labels = {n: f.set_index("TransactionID")["isFraud"].astype("int8") for n, f in raw.items() if "isFraud" in f}

for a, b in [("train", "validation"), ("validation", "holdout"), ("holdout", "kaggle_test")]:
    assert raw[a]["TransactionDT"].max() < raw[b]["TransactionDT"].min(), f"{a} overlaps {b}"
if CONFIG["expected_train_rows"] is not None:
    assert len(raw["train"]) == CONFIG["expected_train_rows"], "training period differs from the EDA"

for n, f in raw.items():
    fraud = f"fraud rate {f['isFraud'].mean():.4f}" if "isFraud" in f else "no labels"
    print(f"{n:12s} {len(f):>8,} rows, days {f['TransactionDT'].min() // SECONDS_PER_DAY:.0f} "
          f"to {f['TransactionDT'].max() // SECONDS_PER_DAY:.0f}, {fraud}")

train         472,432 rows, days 1 to 141, fraud rate 0.0351
validation     59,054 rows, days 141 to 161, fraud rate 0.0313
holdout        59,054 rows, days 161 to 182, fraud rate 0.0375
kaggle_test   506,691 rows, days 213 to 395, no labels


## 3. Helper columns

These are building blocks for later sections. `D1n` (day minus `D1`, the EDA's estimate of when the card was first used), the normalized email domain, the email provider, and the device family are needed to build keys and encodings.

The device rules are the EDA's final rules (cell 124, also used by Jean), with Eric's iPhone, iPad, and "mac os" rules added before the general Android rule. One fix to the EDA rules: a bare `rv:11.0` is Internet Explorer 11, which only runs on Windows, so it now maps to `windows` instead of `firefox` (1,651 training transactions). Other `rv:` tags are Firefox versions, so `firefox` there means a computer running Firefox with an unknown operating system. A missing `DeviceInfo` stays missing here, so it never counts as a device in Section 7.

In [4]:
DEVICE_RULES = [
    ("windows", "windows"), ("ios", "ios"), ("macos", "macos"),
    ("iphone", "ios"), ("ipad", "ios"), ("mac os", "macos"),          # from Eric's notebook
    ("trident", "windows"),
    ("sm-", "samsung"), ("samsung", "samsung"), ("gt-", "samsung"),
    ("moto", "motorola"), ("xt1", "motorola"),
    ("lg", "lg"), ("huawei", "huawei"), ("ale-", "huawei"), ("hi6210sft", "huawei"),
    ("pixel", "google"), ("nexus", "google"), ("redmi", "xiaomi"), ("mi ", "xiaomi"),
    ("htc", "htc"), ("lenovo", "lenovo"), ("zte", "zte"), ("blade", "zte"),
    ("linux", "linux"),
    ("rv:11.0", "windows"),   # Internet Explorer 11, which only runs on Windows
    ("rv:", "firefox"),       # other rv: tags are Firefox versions; operating system unknown
    ("build/", "android_other"),
]

def family_of(text):
    text = str(text).strip().lower()
    if not text:
        return np.nan
    return next((fam for key, fam in DEVICE_RULES if key in text), "other")

def normalize_domain(s):
    return s.astype("string").str.strip().str.lower().replace("", pd.NA)

for n, f in raw.items():
    f["P_emaildomain"] = normalize_domain(f["P_emaildomain"])
    f["R_emaildomain"] = normalize_domain(f["R_emaildomain"])
    f["P_provider"] = f["P_emaildomain"].str.split(".").str[0]
    uniq = f["DeviceInfo"].dropna().unique()
    f["device_family"] = f["DeviceInfo"].map(dict(zip(uniq, map(family_of, uniq))))
    f["D1n"] = f["TransactionDT"] // SECONDS_PER_DAY - f["D1"]

def cat_key(s):
    """Categories as text, with missing values as their own category."""
    return s.astype("string").fillna(MISSING)

print(raw["train"]["device_family"].value_counts(dropna=False).head(12).to_string())

device_family
NaN              372896
windows           47826
ios               17084
macos             10843
samsung            9915
motorola           2717
android_other      2448
firefox            2159
lg                 2111
huawei             2033
other               883
zte                 409


## 4. Route each categorical column

The advisor warned that one-hot encoding a column with many categories creates sparse, wide data. So each candidate column is routed by how many categories it has in training. Columns with `onehot_max_levels` or fewer are handled natively by tree models and one-hot encoded for the neural net. Columns with more get frequency encoding instead (Section 6), and the stable ones also get target (mean) encoding in Section 10b.

**`id_14` (from Salvador's notebook).** `id_14` is a timezone offset stored as a number, such as −300 or 60. Salvador's notebook, drawing on Kaggle discussions, treats it as a category rather than a quantity: a timezone is a place, not an amount. It is added to the candidates here. It has more than 10 values, so it gets frequency encoding (`freq_id_14`). The tree table also keeps the original number, since its order is meaningful geographically (neighboring offsets are neighboring regions); the neural-net table drops the raw number and uses only the frequency column, like the other many-category columns.

### The math

**Number of categories.** For a candidate column $c$, count its distinct non-missing values in the training period:

$$L(c) = \lvert \{\, v_i(c) : i \in \text{train},\ v_i(c) \text{ not missing} \,\} \rvert$$

**Routing rule.** With $L_{\max}$ = `onehot_max_levels` = 10:

$$\text{route}(c) =
\begin{cases}
\text{few categories} & \text{if } L(c) \le L_{\max} \\
\text{many categories} & \text{if } L(c) > L_{\max}
\end{cases}$$

**What each route produces.**

| Route | Tree table | Neural-net table |
|---|---|---|
| Few categories | the column itself, as a category | one-hot: one 0/1 column per category |
| Many categories | the column itself, plus 1 frequency column | 1 frequency column only |

**Why the cutoff matters.** One-hot encoding turns a column into one 0/1 column per category, and each row has a 1 in exactly one of them:

$$\text{onehot}_k(v_i) = \mathbb{1}[\,v_i = k\,] \quad \text{for each category } k$$

So one-hot adds about $L(c)$ columns, while frequency encoding adds 1. For the 13 many-category columns here, one-hot would add about $\sum_c L(c) \approx 14{,}400$ columns (12,730 from `card1` alone), almost all zeros. Frequency encoding adds 13.

The one-hot columns in the neural-net table also include one column for `__MISSING__` and one for `__UNSEEN__` (a value never seen in training), so a few-category column becomes $L(c) + 2$ columns.

In [5]:
train = raw["train"]
y_train = train["isFraud"].to_numpy()

candidates = [c for c in CONFIG["encode_candidates"] if c in train.columns]
n_levels = {c: train[c].nunique(dropna=True) for c in candidates}
low_card = [c for c in candidates if n_levels[c] <= CONFIG["onehot_max_levels"]]
high_card = [c for c in candidates if n_levels[c] > CONFIG["onehot_max_levels"]]
print("Few categories (native for trees, one-hot for the neural net):", low_card)
print("Many categories (frequency encoding):", high_card)

Few categories (native for trees, one-hot for the neural net): ['ProductCD', 'card4', 'card6', 'DeviceType', 'M4']
Many categories (frequency encoding): ['card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain', 'P_provider', 'device_family', 'id_14', 'id_30', 'id_31', 'id_33']


## 6. Frequency encoding for every many-category column

Each category becomes its share of training transactions; missing values get their own share and categories never seen in training get 0. Both teammates did this (Jean with counts, Eric with shares). It is repeated here for all many-category columns because the neural-net table needs a numeric version of every column it can't one-hot encode.

### Why frequency encoding

Frequency encoding tells the model how common or rare each value is, which it can't easily get from the raw column.

1. **The raw codes aren't real numbers.** `card1` has 12,730 codes, and code 13926 isn't "more" than code 2755. A tree splitting on raw `card1` (for example, "card1 > 10,000") groups unrelated cards by accident. `freq_card1` gives the column a meaningful order, from rarely used cards to constantly used ones.
2. **The model can't learn thousands of categories one by one.** Most card codes appear only a few times, too few to learn anything about each individually. The model can instead learn a pattern about groups ("cards used this rarely behave like this") and apply it to every card in the group.
3. **Rarity can carry fraud signal.** Jean's notebook split transactions into 10 groups by `card1` frequency, and fraud rates ranged from 2.6% to 5.1% across the groups. The pattern was not simply "rarer means riskier": the rarest group had one of the lowest fraud rates. Tree models can pick up uneven patterns like this.
4. **It handles values never seen in training.** A raw code the model never saw tells it nothing. A frequency of 0 says something real: this value is brand new.
5. **The neural net needs it.** One-hot encoding `card1` would add 12,730 columns. One frequency column is the compact replacement.
6. **It is safe.** It never uses fraud labels, so there is no leakage risk and no need for folds or smoothing.

Like every feature here, it is a candidate. The modeling pipeline will show whether the `freq_` columns actually improve results.

### The math

**Notation.** For a column $c$ and a category $k$: $N$ is the number of training transactions, and $n(c, k)$ is how many of them have $c = k$. Missing values are treated as their own category, $k = \texttt{\_\_MISSING\_\_}$.

**Training frequency of each category.**

$$\text{freq}(c, k) = \frac{n(c, k)}{N}$$

**Encoding a transaction** $i$ in any period (train, validation, holdout, or Kaggle test):

$$\text{freq\_}c_i =
\begin{cases}
\text{freq}(c, v_i) & \text{if } v_i \text{ appeared in training} \\
\text{freq}(c, \texttt{\_\_MISSING\_\_}) & \text{if } v_i \text{ is missing} \\
0 & \text{if } v_i \text{ never appeared in training}
\end{cases}$$

The frequencies are computed once from training and never updated, so validation and test values can't leak into them.

**Example** (illustrative numbers). If 180,000 of the $N = 472{,}432$ training transactions have `P_emaildomain = gmail.com`:

$$\text{freq}(\text{P\_emaildomain}, \text{gmail.com}) = \frac{180{,}000}{472{,}432} \approx 0.381$$

Every transaction from gmail.com, in any period, gets 0.381.

**Counts vs. shares.** Jean's version stores $n(c, k)$ and Eric's stores $n(c, k) / N$. Dividing every value by the same $N$ keeps their order, so tree models make the same splits either way. Shares are used here because they stay comparable if the size of the training set ever changes, and because they are already on a 0 to 1 scale for the neural net.

**No labels are used.** Frequency encoding only counts how common each category is. It never looks at `isFraud`, so unlike mean encoding it needs no folds or smoothing, and a category's value can't carry outdated fraud rates.

In [6]:
freq_maps = {c: cat_key(train[c]).value_counts(normalize=True) for c in high_card}

def frequency_encode(frame):
    return pd.DataFrame({f"freq_{c}": cat_key(frame[c]).map(freq_maps[c]).fillna(0).astype("float32")
                         for c in high_card}, index=frame.index)

## 6b. Amount, weekday, and email relationship (from Eric)

These features come from Eric's notebook. Two of them carry out decisions from the EDA decision log (Branch 3, H1): amount relative to its product's typical amount, and amount-frequency features. Every reference value is learned from training only.

### Amount features

**`amount_to_product_median`: is this amount large for its product?** The EDA found fraud amounts are higher than legitimate ones *within* every product, but products have very different typical amounts. Comparing each amount with its own product's training median puts them on one scale:

$$\text{amount\_to\_product\_median} = \frac{\text{amount}}{\max(\text{training median amount for this product},\ 1)}$$

Example: if product W's training median is 80, a 400 payment on W gets 400 ÷ 80 = **5.0**, five times typical. A product never seen in training uses the overall training median. The floor of 1 avoids dividing by tiny numbers.

**`amount_train_frequency`: how common is this exact amount?** The EDA found legitimate amounts cluster at common price points, like fixed subscription prices. Each amount is rounded to the cent and replaced by its share of training transactions, the same idea as Section 6:

$$\text{amount\_train\_frequency} = \frac{\text{training transactions with this amount (to the cent)}}{\text{all training transactions}}$$

An amount never seen in training gets 0.

**`amount_fractional_unit`: the part after the decimal point.** 25.375 gives 0.375. The EDA suggested testing a decimal-places feature later; this is it.

**`amount_has_subcent_precision`: does the amount go beyond whole cents?** 25.37 gives 0; 25.375 gives 1. This describes how the amount was recorded, which can differ by payment source. Salvador's notebook flags the same thing (three decimal places), noting from Kaggle discussions that it often points to a currency conversion, which can mean a foreign or cross-border transaction.

### Weekday features

**`weekday_sin`, `weekday_cos`:** the same circle idea as `hour_sin` and `hour_cos` in Section 10, for a 7-day cycle:

$$\text{weekday} = \left\lfloor \frac{\text{TransactionDT}}{86{,}400} \right\rfloor \bmod 7, \qquad \text{weekday\_sin} = \sin\!\left(\frac{2\pi \cdot \text{weekday}}{7}\right), \qquad \text{weekday\_cos} = \cos\!\left(\frac{2\pi \cdot \text{weekday}}{7}\right)$$

The starting day is unknown, so weekday 0 is not confirmed to be Monday. The EDA found little day-of-week signal (fraud between 3.2% and 3.8%), so these are low priority; they are included so the comparison can confirm it.

### Email relationship

**`email_domain_relationship`:** compares the purchaser's and recipient's email domains. Five possible values: `same_domain`, `different_domains`, `purchaser_missing`, `recipient_missing`, `both_missing`. It is stored as text so Section 10 treats it as a category. Matching domains does not mean matching email addresses: many people share `gmail.com`. The EDA found this mostly repeats product information, so it is also low priority.

In [7]:
# Product medians and amount frequencies, learned from training only
product_medians = train.groupby("ProductCD")["TransactionAmt"].median()
global_amount_median = float(train["TransactionAmt"].median())

def cents_key(amount):
    """Amount rounded to whole cents, as an integer (10.004 and 10.00 both become 1000)."""
    return np.rint(amount.astype("float64") * 100).astype("int64")

amount_freq_map = cents_key(train["TransactionAmt"]).value_counts(normalize=True)

def amount_time_features(frame):
    amount = frame["TransactionAmt"].astype("float64")
    weekday = frame["TransactionDT"] // SECONDS_PER_DAY % 7
    typical = frame["ProductCD"].map(product_medians).fillna(global_amount_median).clip(lower=1)
    return pd.DataFrame({
        "amount_to_product_median": (amount / typical).astype("float32"),
        "amount_train_frequency": cents_key(amount).map(amount_freq_map).fillna(0).astype("float32"),
        "amount_fractional_unit": (amount - np.floor(amount)).astype("float32"),
        "amount_has_subcent_precision": (~np.isclose(amount * 100, np.round(amount * 100),
                                                     rtol=0, atol=1e-6)).astype("int8"),
        "weekday_sin": np.sin(2 * np.pi * weekday / 7).astype("float32"),
        "weekday_cos": np.cos(2 * np.pi * weekday / 7).astype("float32"),
    }, index=frame.index)

# Email relationship, added as a text column so Section 10 treats it as a category
for f in raw.values():
    p, r = f["P_emaildomain"], f["R_emaildomain"]
    rel = pd.Series("different_domains", index=f.index, dtype="string")
    rel[p.eq(r).fillna(False).astype(bool)] = "same_domain"
    rel[p.isna().to_numpy()] = "purchaser_missing"
    rel[r.isna().to_numpy()] = "recipient_missing"
    rel[(p.isna() & r.isna()).to_numpy()] = "both_missing"
    f["email_domain_relationship"] = rel

print("Training median amount by product:")
print(product_medians.round(2).to_string())
print("\nEmail relationship (training):")
print(train["email_domain_relationship"].value_counts().to_string())

Training median amount by product:
ProductCD
C     31.66
H     50.00
R    125.00
S     30.00
W     78.97

Email relationship (training):
email_domain_relationship
recipient_missing    292877
same_domain           85848
both_missing          65935
different_domains     20121
purchaser_missing      7651


## 7. Account history with a second key for address-missing transactions

**The gap.** The EDA found that transactions with no `addr1` are 11.4% of training transactions but 37% of all fraud. Both teammates' account keys need `addr1`, so these transactions get no account history at all, even though the EDA also found fraud clusters inside accounts.

**Why account keys at all.** Fraud often shows up as a change in a customer's own behavior: a much larger purchase than usual, a new device, or several purchases close together. Measuring that requires knowing which transactions belong to the same customer, and the data has no customer ID. `card1` alone is shared by many people, so the keys combine it with other columns to get closer to one customer.

**Two keys.**

- **Key 1** (`k1_`): `card1 | addr1 | D1n`, the EDA's key and the same rule Eric uses. Missing if any part is missing.
- **Key 2** (`k2_`): `card1 | P_emaildomain | D1n`. It needs an email domain instead of an address, so it can link many of the address-missing transactions. It is also a broader key (many people share `gmail.com`), which is why it is kept separate instead of replacing Key 1.

**Time rules.** Only transactions with a strictly earlier timestamp count as history, and fraud labels are never used. Payments at the same second can't see each other. Validation, holdout, and Kaggle payments enter the activity log for later payments (without labels), simulating a system that updates as transactions arrive. This follows Eric's design and reuses his approach for the count, mean, and gap features.

### Building the key

`D1n` estimates the day the card was first used. `TransactionDT` is in seconds, so dividing by 86,400 (seconds in a day) and rounding down gives the day number:

$$D1n_i = \left\lfloor \frac{t_i}{86{,}400} \right\rfloor - D1_i$$

The key is the three values joined as text. For example, `card1` = 1234, `addr1` = 315, `D1n` = 150 gives the Key 1 label `1234|315.0|150.0`. All transactions with the same label are treated as one account.

### A worked example

Every feature below is explained using this one Key 1 account. Times are in seconds.

| Transaction | Time | Amount | Email | Device |
|---|---|---|---|---|
| T1 | 1,000 | \$50 | gmail.com | windows |
| T2 | 5,000 | \$40 | gmail.com | windows |
| T3 | 9,000 | \$200 | yahoo.com | samsung |
| T4 | 9,000 | \$60 | gmail.com | windows |
| T5 | 20,000 | \$500 | hotmail.com | ios |

**The prior set.** For each transaction $i$, its history is every transaction from the same account with an earlier time:

$$P(i) = \{\, j : \text{same account as } i,\ t_j < t_i \,\}$$

For T5, the prior set is T1, T2, T3, and T4. For T3, it is only T1 and T2. T4 happened at the same second, so it is not "earlier."

### The features

Each feature is calculated once for Key 1 and once for Key 2, giving the `k1_` and `k2_` columns.

**`key_missing`: could the account key be built?**
1 if any part of the key is missing, otherwise 0. Transactions without a key get no history.

**`prior_count`: how many earlier transactions did this account have?**

$$\text{prior\_count}_i = \text{number of transactions in } P(i)$$

For T5: 4 (T1 to T4). For T3 and T4: 2 each. For T1: 0.

**`has_prior`: is there any history at all?**
1 if `prior_count` is above 0, otherwise 0. For T1: 0. For every other transaction: 1.

**`prior_mean_amt`: what did this account usually spend?**

$$\text{prior\_mean\_amt}_i = \frac{\text{sum of earlier amounts}}{\text{prior\_count}_i}$$

For T5: (50 + 40 + 200 + 60) ÷ 4 = **87.50**. For T1 there is nothing to average, so it is left missing.

**`amt_to_prior_mean`: how does this amount compare with the usual?**

$$\text{amt\_to\_prior\_mean}_i = \frac{\text{amount}_i}{\max(\text{prior\_mean\_amt}_i,\ 1)}$$

For T5: 500 ÷ 87.50 = **5.71**, almost six times the usual amount. The $\max(\ldots, 1)$ treats any average below \$1 as \$1, so a tiny earlier average (say \$0.10) can't produce an extreme ratio. Missing when there is no history.

**`secs_since_prev`: how long since the account's last transaction?**

$$\text{secs\_since\_prev}_i = t_i - \text{time of the most recent transaction in } P(i)$$

For T5: 20,000 − 9,000 = **11,000** seconds. Missing when there is no history.

**`prior_distinct_P_emaildomain` (Key 1) and `prior_distinct_addr1` (Key 2): how many different emails or addresses had the account used before?**

$$\text{prior\_distinct}_i = \text{number of different non-missing values among the transactions in } P(i)$$

For T5's emails: T1 to T4 used gmail.com and yahoo.com, so **2**. Hotmail is new at T5, so it isn't counted. Key 2 counts addresses instead of emails, because the email is already part of Key 2.

**`prior_distinct_device_family`: how many different devices had the account used before?**
Same formula with device family. For T5: windows and samsung, so **2**. iOS is new at T5.

**All five transactions together:**

| | prior_count | prior_mean_amt | amt_to_prior_mean | secs_since_prev | distinct emails | distinct devices |
|---|---|---|---|---|---|---|
| T1 | 0 | missing | missing | missing | 0 | 0 |
| T2 | 1 | 50 | 0.80 | 4,000 | 1 | 1 |
| T3 | 2 | 45 | 4.44 | 4,000 | 1 | 1 |
| T4 | 2 | 45 | 1.33 | 4,000 | 1 | 1 |
| T5 | 4 | 87.50 | 5.71 | 11,000 | 2 | 2 |

T5 is the kind of transaction these features are designed to catch: almost six times the usual amount, from an email and a device the account had never used.

**Why "no history" is not zero.** Counts are 0 when there is no history, because zero earlier transactions is a true count. The average, ratio, and time gap are left missing, because 0 would wrongly mean "\$0 usual spend" or "0 seconds since the last purchase."

### How the code does this quickly

Searching each transaction's history one by one would be very slow on a million transactions. Instead, the code groups transactions by account and exact timestamp, keeps a running total over time within each account, and subtracts the current moment. For T3 and T4, the running count up to time 9,000 is 4 (T1, T2, T3, T4). Subtracting the 2 transactions at 9,000 leaves 2, which is the correct prior count for both. Subtracting the whole moment, not just one transaction, is what keeps simultaneous payments from seeing each other.

For the distinct counts, the code finds the first time the account used each value (gmail at 1,000, yahoo at 9,000, hotmail at 20,000) and counts how many of those first uses happened before the current transaction.

In [8]:
def make_key(frame, parts):
    p = frame[parts]
    complete = p.notna().all(axis=1)
    key = pd.Series(pd.NA, index=frame.index, dtype="string")
    s = p.loc[complete].astype("string")
    key.loc[complete] = s[parts[0]].str.cat([s[c] for c in parts[1:]], sep="|")
    return key

KEYS = {
    "k1": (["card1", "addr1", "D1n"], ["P_emaildomain", "device_family"]),
    "k2": (["card1", "P_emaildomain", "D1n"], ["addr1", "device_family"]),
}

def strict_past(events, key, distinct_cols, prefix):
    """History from strictly earlier timestamps only. events is indexed by TransactionID."""
    elig = events[events[key].notna()]
    bt = elig.groupby([key, "TransactionDT"], sort=True).agg(
        n=("TransactionAmt", "size"), amt=("TransactionAmt", "sum")).reset_index()
    g = bt.groupby(key, sort=False)
    bt["prior_count"] = g["n"].cumsum() - bt["n"]
    bt["prior_mean_amt"] = (g["amt"].cumsum() - bt["amt"]) / bt["prior_count"].replace(0, np.nan)
    bt["secs_since_prev"] = bt["TransactionDT"] - g["TransactionDT"].shift(1)
    out_cols = ["prior_count", "prior_mean_amt", "secs_since_prev"]
    for d in distinct_cols:
        # first time each (account, value) pair appears; count values first seen strictly earlier
        first = elig.dropna(subset=[d]).groupby([key, d])["TransactionDT"].min().reset_index()
        new = first.groupby([key, "TransactionDT"]).size().rename("new")
        bt = bt.join(new, on=[key, "TransactionDT"])
        bt["new"] = bt["new"].fillna(0)
        bt[f"prior_distinct_{d}"] = bt.groupby(key, sort=False)["new"].cumsum() - bt["new"]
        bt = bt.drop(columns="new")
        out_cols.append(f"prior_distinct_{d}")
    res = events[[key, "TransactionDT", "TransactionAmt"]].join(
        bt.set_index([key, "TransactionDT"])[out_cols], on=[key, "TransactionDT"])
    res["key_missing"] = res[key].isna().astype("int8")
    for c in out_cols:
        if c.startswith("prior_count") or c.startswith("prior_distinct"):
            res[c] = res[c].fillna(0).astype("float32")
    res["has_prior"] = (res["prior_count"] > 0).astype("int8")
    res["amt_to_prior_mean"] = (res["TransactionAmt"] / res["prior_mean_amt"].clip(lower=1)).astype("float32")
    keep = ["key_missing", "has_prior", "amt_to_prior_mean"] + out_cols
    return res[keep].add_prefix(f"{prefix}_")

keys = {}
for n, f in raw.items():
    keys[n] = pd.DataFrame({name: make_key(f, parts).to_numpy() for name, (parts, _) in KEYS.items()},
                           index=pd.Index(f["TransactionID"], name="TransactionID"))

distinct_needed = sorted({d for _, (_, ds) in KEYS.items() for d in ds})
events = pd.concat([
    pd.concat([f[["TransactionID", "TransactionDT", "TransactionAmt"] + distinct_needed].set_index("TransactionID"), keys[n]], axis=1)
    for n, f in raw.items()
])
assert events.index.is_unique
history = pd.concat([strict_past(events, k, ds, k) for k, (_, ds) in KEYS.items()], axis=1)
del events
gc.collect()
print(history.columns.tolist())

['k1_key_missing', 'k1_has_prior', 'k1_amt_to_prior_mean', 'k1_prior_count', 'k1_prior_mean_amt', 'k1_secs_since_prev', 'k1_prior_distinct_P_emaildomain', 'k1_prior_distinct_device_family', 'k2_key_missing', 'k2_has_prior', 'k2_amt_to_prior_mean', 'k2_prior_count', 'k2_prior_mean_amt', 'k2_secs_since_prev', 'k2_prior_distinct_addr1', 'k2_prior_distinct_device_family']


**Coverage check (training labels only).** How many address-missing transactions does Key 2 reach, and does having history there say anything about fraud?

### How the numbers are calculated

**Groups.** Every training transaction goes into one of two groups: **address missing** (`addr1` is blank) or **address present**.

**Shares in the first table.** Each column except `isFraud` is a yes/no flag, stored as True/False. Taking the average of a True/False column gives the share of rows that are True, because True counts as 1 and False as 0. For example, if a group had 4 transactions and Key 2 could be built for 3 of them:

$$\text{k2\_has\_key} = \frac{1 + 1 + 1 + 0}{4} = 0.75 \quad (75\%)$$

**Fraud rate.** `isFraud` is 1 for fraud and 0 for legitimate, so its average is the fraud rate:

$$\text{fraud rate} = \frac{\text{number of fraud transactions in the group}}{\text{number of transactions in the group}}$$

For example, 3 fraud cases among 25 transactions gives 3 ÷ 25 = 0.12, or 12%.

### Results from the full run

**First table** (shares of each group):

| Group | Key 1 can be built | Key 2 can be built | Key 1 history | Key 2 history | Fraud rate |
|---|---|---|---|---|---|
| Address missing | 0% | 95.7% | 0% | 58.3% | 11.4% |
| Address present | 99.8% | 82.8% | 59.9% | 49.3% | 2.5% |

- Key 1 can't be built for any address-missing transaction, as expected, since the address is part of Key 1.
- Key 2 can be built for 95.7% of them, because they still have an email domain.
- **58.3% of address-missing transactions now have account history through Key 2.** Before Key 2 they had none. That's about 31,000 training transactions in the highest-risk group.
- Key 2 can be built for 82.8% of address-present transactions. The rest are missing an email domain.

**Second table** (address-missing transactions only, split by Key 2 history):

| Key 2 history? | Transactions | Fraud rate | Fraud cases |
|---|---|---|---|
| No | 22,395 | 10.65% | about 2,385 |
| Yes | 31,366 | 11.93% | about 3,742 |

Fraud cases are the fraud rate times the number of transactions, for example 0.1193 × 31,366 ≈ 3,742. The two groups together give (2,385 + 3,742) ÷ 53,761 ≈ 11.4%, matching the address-missing fraud rate in the first table.

**How to read it.** Transactions with Key 2 history are slightly more likely to be fraud (11.93% vs. 10.65%). With groups this large the gap is real, but it's small. That doesn't mean Key 2 is weak. `has_prior` is only a yes/no flag. What should matter more is *what* the history shows: an amount far above the usual, a new device or email, or a purchase minutes after the last one. Section 13's signal check scores those features individually, and the modeling pipeline will show whether the `k2_` features improve results overall.

**Note for comparing with the EDA.** The EDA found 67.7% of training transactions in repeat accounts, while 59.9% of address-present transactions have Key 1 history here. The EDA counted every transaction in a repeat account, including its first. Here an account's first transaction has no earlier history, so it isn't counted.

In [9]:
h_tr = history.loc[train["TransactionID"]].set_axis(train.index)
no_addr = train["addr1"].isna()
cov = pd.DataFrame({
    "group": np.where(no_addr, "address missing", "address present"),
    "k1_has_key": h_tr["k1_key_missing"].eq(0), "k2_has_key": h_tr["k2_key_missing"].eq(0),
    "k1_has_prior": h_tr["k1_has_prior"].eq(1), "k2_has_prior": h_tr["k2_has_prior"].eq(1),
    "isFraud": y_train,
})
print(cov.groupby("group").mean().round(3).to_string())

sub = cov[cov["group"] == "address missing"]
print("\nAddress-missing transactions, fraud rate by Key 2 history:")
print(sub.groupby("k2_has_prior")["isFraud"].agg(n="size", fraud_rate="mean").round(4).to_string())
del h_tr, cov, sub

                 k1_has_key  k2_has_key  k1_has_prior  k2_has_prior  isFraud
group                                                                       
address missing       0.000       0.957         0.000         0.583    0.114
address present       0.998       0.828         0.599         0.493    0.025

Address-missing transactions, fraud rate by Key 2 history:
                  n  fraud_rate
k2_has_prior                   
False         22395      0.1065
True          31366      0.1193


## 8. One missing indicator per missingness block

**What a block is.** The EDA found groups of columns that are always missing together: whenever one column in the group is missing for a transaction, every other column in the group is missing too. These groups are called **blocks**. They likely come from separate data sources: if a source recorded nothing for a transaction, all of its columns are empty at once.

**What this section does.** The EDA found the blocks and decided that one missing indicator per block is enough, but it never built those indicators. This section builds them.

### How it works

**Step 1: Find candidate blocks.** Count the missing values in each column of the training data. Columns with exactly the same count are grouped together, the same method the EDA used.

**Step 2: Confirm each block.** Having the same count isn't enough, so the code checks that the columns are missing in exactly the **same rows**. If any group fails, the notebook stops with an error.

**Step 3: Create one flag per block.** Each block gets one new 0/1 column:

$$\text{na\_block}_i = \begin{cases} 1 & \text{if the block's data is missing for transaction } i \\ 0 & \text{otherwise} \end{cases}$$

**Step 4: Apply it to later periods.** Blocks are defined from training only. For validation, holdout, and Kaggle test, each block's flag follows the block's **first column**: if that column is missing, the flag is 1. This assumes the block's columns keep going missing together, which they should if they come from the same data source.

### Example

Columns V1, V2, and V3 form a block. C1 is a separate column.

| Transaction | V1 | V2 | V3 | C1 |
|---|---|---|---|---|
| T1 | 0.5 | 1.2 | 3.0 | 2 |
| T2 | missing | missing | missing | 1 |
| T3 | 0.8 | 0.9 | 2.1 | missing |

- **Step 1:** V1, V2, V3, and C1 each have 1 missing value, so all four could be grouped by count alone.
- **Step 2:** V1, V2, and V3 are all missing in T2, so they form a confirmed block. C1 is missing in T3 instead, so it is not part of the block.
- **Step 3:** The block gets one flag, `na_block_00`: 0 for T1, **1 for T2**, 0 for T3. Three identical flags would add nothing, so one is enough.

### Results from the full run

67 missingness groups in training: **24 blocks covering 371 columns**, and **43 single columns** that each go missing in their own pattern. This matches the EDA exactly. The largest block has 46 columns, so its single flag replaces 46 identical ones.

### Where the flags are used

**Tree table: the 24 block flags.** XGBoost and LightGBM handle missing values on their own and can split directly on "missing vs. present," so the flags are a convenience. They let a tree treat a whole block's absence as one signal.

**Neural-net table: the 24 block flags plus one flag per single column** (for example, `na_C1`). Here they matter much more. A neural net can't accept missing values, so they are filled with the training median. After filling, T2's V1 looks like an ordinary number, and the flag is the only trace that it was missing. That matters because the EDA found that missingness patterns carry fraud signal.

In [10]:
null_counts = train[raw_cols].isna().sum()
null_counts = null_counts[null_counts > 0]
groups = [list(g.index) for _, g in null_counts.groupby(null_counts)]
blocks = [g for g in groups if len(g) > 1]
singles = [g[0] for g in groups if len(g) == 1]

for cols in blocks:
    na = train[cols].isna()
    assert na.eq(na.iloc[:, 0], axis=0).all().all(), f"block starting {cols[0]} is not row-identical"

block_names = {f"na_block_{i:02d}": cols for i, cols in enumerate(sorted(blocks, key=lambda c: -len(c)))}

def block_flags(frame):
    return pd.DataFrame({name: frame[cols[0]].isna().astype("int8") for name, cols in block_names.items()},
                        index=frame.index)

print(f"{len(groups)} missingness groups: {len(blocks)} blocks covering {sum(map(len, blocks))} columns, "
      f"{len(singles)} single columns")

67 missingness groups: 24 blocks covering 371 columns, 43 single columns


In [11]:
for name, cols in block_names.items():
    print(f"{name}: {len(cols)} columns, e.g. {cols[:5]}")

na_block_00: 46 columns, e.g. ['V217', 'V218', 'V219', 'V223', 'V224']
na_block_01: 43 columns, e.g. ['V95', 'V96', 'V97', 'V98', 'V99']
na_block_02: 32 columns, e.g. ['V279', 'V280', 'V284', 'V285', 'V286']
na_block_03: 31 columns, e.g. ['V167', 'V168', 'V172', 'V173', 'V176']
na_block_04: 29 columns, e.g. ['V138', 'V139', 'V140', 'V141', 'V142']
na_block_05: 23 columns, e.g. ['V12', 'V13', 'V14', 'V15', 'V16']
na_block_06: 22 columns, e.g. ['V53', 'V54', 'V55', 'V56', 'V57']
na_block_07: 20 columns, e.g. ['V75', 'V76', 'V77', 'V78', 'V79']
na_block_08: 19 columns, e.g. ['V169', 'V170', 'V171', 'V174', 'V175']
na_block_09: 18 columns, e.g. ['V35', 'V36', 'V37', 'V38', 'V39']
na_block_10: 18 columns, e.g. ['V322', 'V323', 'V324', 'V325', 'V326']
na_block_11: 16 columns, e.g. ['V220', 'V221', 'V222', 'V227', 'V234']
na_block_12: 12 columns, e.g. ['D1', 'V281', 'V282', 'V283', 'V288']
na_block_13: 12 columns, e.g. ['D11', 'V1', 'V2', 'V3', 'V4']
na_block_14: 8 columns, e.g. ['id_11', 'id

## 8b. Missingness counts (Eric) and missingness clusters (Jean)

Section 8 flags **which** blocks are missing. These features describe missingness from two other angles.

### Missingness counts (from Eric's notebook)

How much information is missing for each transaction, counted over the original columns:

| Feature | Counts missing values in |
|---|---|
| `raw_missing_count` | all original input columns |
| `identity_missing_count` | identity and device columns (`id_*`, `DeviceType`, `DeviceInfo`) |
| `card_address_missing_count` | card and address columns (`card*`, `addr*`) |
| `v_missing_count` | the anonymous V columns |

Example: a transaction missing 230 of its original inputs, 40 of them identity columns, gets `raw_missing_count` = 230 and `identity_missing_count` = 40. A blank does not automatically mean fraud: different products collect different information. The model learns whether the amount of missing information matters together with the other inputs.

### Missingness clusters (Jean's approach, from the EDA)

The EDA (Branch 2, H5) grouped transactions by their overall **pattern** of missing information using k-means clustering. It found 3 clusters that roughly match identity and product types, with fraud rates of about 2.1%, 4.4%, and 12.0%. Jean turned this into a reusable feature.

**How it works.**

1. Take one representative column from each missingness group in Section 8 (all 67 groups).
2. For each transaction, write down 1 or 0 for whether each representative is missing. That gives each transaction a 67-digit missingness pattern.
3. **k-means with k = 3** sorts the training transactions into 3 groups of similar patterns. It places 3 group centers so that each transaction is as close as possible to its own group's center. k = 3 comes from the EDA, where it scored best (silhouette 0.55).
4. Each transaction in every period is assigned to the nearest center. The centers are learned from training only.

**`na_cluster`** is the cluster number: 0, 1, or 2. The numbers are labels, not amounts (cluster 2 is not "more" than cluster 1), so they are stored as text and treated as a category. The cell prints each cluster's size and training fraud rate. The numbering is arbitrary and can change between runs; read the clusters by their fraud rates, not their numbers.

### What was not added from Eric's notebook

Eric's and Susie's single-column missing flags (`addr1_missing`, `D1_missing`, the email-missing flags, and others) and his categorical `relative_hour` and `relative_weekday` are left out because they are already covered: tree models see missing values directly, the neural-net table gets a missing flag for every block and single column (Section 11), and hour and weekday are encoded as sine and cosine.

In [12]:
from sklearn.cluster import MiniBatchKMeans

# Missingness counts (Eric's definitions)
v_cols = [c for c in raw_cols if c.startswith("V") and c[1:].isdigit()]
identity_cols = [c for c in raw_cols if c.startswith("id_")] + ["DeviceType", "DeviceInfo"]
card_addr_cols = [c for c in raw_cols if c.startswith(("card", "addr"))]

def missing_counts(frame):
    return pd.DataFrame({
        "raw_missing_count": frame[raw_cols].isna().sum(axis=1).astype("int16"),
        "identity_missing_count": frame[identity_cols].isna().sum(axis=1).astype("int16"),
        "card_address_missing_count": frame[card_addr_cols].isna().sum(axis=1).astype("int16"),
        "v_missing_count": frame[v_cols].isna().sum(axis=1).astype("int16"),
    }, index=frame.index)

# Missingness profile clusters (Jean's approach, from the EDA's k-means result)
rep_cols = [g[0] for g in groups]   # one representative column per missingness group (Section 8)

def na_indicators(frame):
    return frame[rep_cols].isna().astype("int8")

na_km = MiniBatchKMeans(n_clusters=3, random_state=42, n_init=5, batch_size=4096).fit(na_indicators(train))
for f in raw.values():
    # cluster numbers are labels, not amounts, so they are stored as text and treated as categories
    f["na_cluster"] = pd.Series(na_km.predict(na_indicators(f)), index=f.index).astype(str)

print("Missingness clusters (training):")
print(train.groupby("na_cluster")["isFraud"].agg(transactions="size", fraud_rate="mean").round(4).to_string())

Missingness clusters (training):
            transactions  fraud_rate
na_cluster                          
0                 354147      0.0213
1                  68414      0.0444
2                  49871      0.1205


## 9. Reduced V-column list (EDA method)

**What it does.** Finds V columns that are near-copies of each other (pairwise multicollinearity) and builds a shorter list without them. This repeats the EDA's H2 follow-up exactly.

**How it works.** On a 100,000-row training sample, within each missingness block (so columns are compared on the same rows):

1. Sort the block's V columns from most distinct values to fewest.
2. Keep the first column.
3. Keep each next column only if its Spearman correlation with every kept column is at most 0.95 in absolute value. Otherwise drop it as a near-copy.

Spearman correlation is the correlation of the columns' ranks. It catches any consistent rising-together relationship, not just straight lines, which matters because many V columns are skewed.

**Example.** In one block, V2 is always exactly twice V1, so their ranks match ($|\rho_S| = 1$) and V2 is dropped. V3's order doesn't match V1's ($|\rho_S| = 0.4$), so V3 is kept.

**Result.** The EDA kept 210 of 339 V columns.

**Where it's used.**
- **Tree table:** all 339 stay. The reduced version is saved as the feature set `all_reduced_v`, following the advisor's rule: keep the smaller set only if performance drops by no more than 1 to 2%.
- **Neural-net table:** uses the 210 by default, as the EDA decided.

**Limit.** It compares columns two at a time, so it won't catch a column that's a combination of others (that would need VIF). The 0.95 threshold is the EDA's judgment call.

In [13]:
v_all = [c for c in raw_cols if c.startswith("V")]
v_sample = train[v_all].sample(min(CONFIG["v_sample_rows"], len(train)), random_state=0)
v_null = train[v_all].isna().sum()
v_keep = []
for _, g in v_null.groupby(v_null):
    cols = list(g.index)
    sub = v_sample[cols].dropna()
    if len(cols) == 1 or len(sub) < 100:
        v_keep += cols
        continue
    corr = sub.corr(method="spearman").abs().fillna(0)
    kept = []
    for c in sub.nunique().sort_values(ascending=False).index:
        if all(corr.loc[c, k] <= CONFIG["v_corr_threshold"] for k in kept):
            kept.append(c)
    v_keep += kept
v_drop = [c for c in v_all if c not in v_keep]
print(f"V columns: {len(v_all)} | kept {len(v_keep)} | dropped {len(v_drop)}")
del v_sample

V columns: 339 | kept 210 | dropped 129


## 10. Tree table

**What this cell does.** It assembles everything built so far into one table per period (train, validation, holdout, Kaggle test) for the tree models (XGBoost, LightGBM). Each row is one transaction, labeled by its `TransactionID`.

### What goes into the table

| Piece | Built in | Columns |
|---|---|---|
| Original number columns (amounts, `card1`, `addr1`, C, D, and V columns, ...) | Section 2 (loaded) | all original numeric inputs |
| Original text columns, converted to categories | this section | all original text inputs, plus `P_provider` and `device_family` from Section 3 (and the two new categories, `email_domain_relationship` and `na_cluster`, counted in their rows below) |
| `has_identity`, `amount_log1p`, `hour_sin`, `hour_cos` | this section | 4 |
| Missing-block flags `na_block_*` | Section 8 | 24 |
| Frequency encodings `freq_*` | Section 6 | 13 |
| Amount and weekday features, `email_domain_relationship` | Section 6b | 7 |
| Missingness counts and `na_cluster` | Section 8b | 5 |
| Account history `k1_*` and `k2_*` | Section 7 | 16 |

Left out: raw `DeviceInfo` (replaced by `device_family`, per the EDA), the helper `D1n`, `TransactionID`, `TransactionDT`, and the fraud label.

The pieces are placed side by side. Each has one row per transaction in the same order, so they line up. The history columns are matched by `TransactionID`, because Section 7 computed them on the combined activity log.

### Text columns become categories

Each text column gets an **allowed list of categories**, learned from training only, plus two special entries:

- `__MISSING__` for blanks
- `__UNSEEN__` for any value that never appeared in training

For example, if a validation transaction has the email domain `newmail.com`, which never appeared in training, it becomes `__UNSEEN__`. Every period uses the same list, so a category has the same internal code everywhere. This follows Eric's approach. Tree models can split on these category columns directly.

Unlike Eric's notebook, `card1`, `card2`, `addr1`, and the other number-coded columns stay numeric here. The advisor flagged columns with thousands of categories as a risk, and their frequency encodings (Section 6) carry how common each code is. Comparing the two choices is a good first experiment for the modeling pipeline.

### Four simple features

- **`has_identity`**: 1 if the transaction has identity data, else 0 (an EDA decision).
- **`amount_log1p`**: $\log(1 + \text{amount})$, which squeezes very large amounts closer together. Trees don't need it, since a log keeps the same order; it's included for the neural net (Section 11) and to match Eric's definition.
- **`hour_sin`, `hour_cos`**: the hour of the day placed on a circle, so 23:00 and 00:00 end up next to each other (an EDA decision). The hour comes from `TransactionDT`:

$$\text{hour} = \left\lfloor \frac{\text{TransactionDT}}{3600} \right\rfloor \bmod 24, \qquad \text{hour\_sin} = \sin\!\left(\frac{2\pi \cdot \text{hour}}{24}\right), \qquad \text{hour\_cos} = \cos\!\left(\frac{2\pi \cdot \text{hour}}{24}\right)$$

### Metadata, kept separately

A second table per period stores `TransactionDT`, `ProductCD`, and both account keys (Section 7). These are **not** model inputs. They're saved so results can later be broken down, for example by product W vs. C, or by accounts seen in training vs. new.

### Same columns, same order

Every period's table is put in training's column order. A model trained on one column order must receive the same order later.

### Feature sets for comparison

The cell also defines named column lists for the modeling pipeline to compare:

| Feature set | What's in it |
|---|---|
| `raw` | original columns only |
| `all_no_history` | everything except the 16 account history columns |
| `all` | everything |
| `all_reduced_v` | everything except the 129 near-duplicate V columns (Section 9) |
| `all_no_jennifer` | added in Section 10i: everything except Jennifer's target encodings and behavior features |

Section 10i updates these sets once Jennifer's columns are added: `all_no_history` then also leaves out her `beh_*` and `vel_*` account-history columns.

These match the team's plan (raw, clean, and history-enriched) plus the advisor's rule to start with all features and reduce only if performance drops by no more than 1 to 2%. The cell prints the total column count and the size of each set.

In [14]:
cat_cols = [c for c in raw_cols + ["P_provider", "device_family", "email_domain_relationship", "na_cluster"]
            if c != "DeviceInfo" and not pd.api.types.is_numeric_dtype(train[c])]
vocab = {c: [MISSING, UNSEEN] + sorted(set(cat_key(train[c])) - {MISSING}) for c in cat_cols}

def as_categories(frame):
    out = {}
    for c in cat_cols:
        k = cat_key(frame[c])
        out[c] = pd.Categorical(k.where(k.isin(vocab[c]), UNSEEN), categories=vocab[c])
    return pd.DataFrame(out, index=frame.index)

def basic_transforms(frame):
    hour = frame["TransactionDT"] // 3600 % 24
    return pd.DataFrame({
        "has_identity": frame["id_01"].notna().astype("int8"),
        "amount_log1p": np.log1p(frame["TransactionAmt"]).astype("float32"),
        "hour_sin": np.sin(2 * np.pi * hour / 24).astype("float32"),
        "hour_cos": np.cos(2 * np.pi * hour / 24).astype("float32"),
    }, index=frame.index)

numeric_raw = [c for c in raw_cols if c != "DeviceInfo" and c not in cat_cols]
tree, meta = {}, {}
for n, f in raw.items():
    parts = [
        f[numeric_raw],
        as_categories(f),
        basic_transforms(f),
        amount_time_features(f),
        missing_counts(f),
        block_flags(f),
        frequency_encode(f),
        history.loc[f["TransactionID"]].set_axis(f.index),
    ]
    X = pd.concat(parts, axis=1)
    X.index = pd.Index(f["TransactionID"], name="TransactionID")
    tree[n] = X
    meta[n] = pd.DataFrame({"TransactionDT": f["TransactionDT"].to_numpy(),
                            "ProductCD": f["ProductCD"].to_numpy()}, index=X.index).join(keys[n])
tree_cols = tree["train"].columns.tolist()
for n in tree:
    tree[n] = tree[n][tree_cols]

new_families = {
    "basic": ["has_identity", "amount_log1p", "hour_sin", "hour_cos"],
    "amount_weekday": ["amount_to_product_median", "amount_train_frequency", "amount_fractional_unit",
                       "amount_has_subcent_precision", "weekday_sin", "weekday_cos"],
    "missing_counts": ["raw_missing_count", "identity_missing_count",
                       "card_address_missing_count", "v_missing_count"],
    "email_device_groups": ["P_provider", "device_family", "email_domain_relationship"],
    "missing_clusters": ["na_cluster"],
    "missing_blocks": list(block_names),
    "frequency": [f"freq_{c}" for c in high_card],
    "history_k1": [c for c in tree_cols if c.startswith("k1_")],
    "history_k2": [c for c in tree_cols if c.startswith("k2_")],
}
history_cols = new_families["history_k1"] + new_families["history_k2"]
feature_sets = {
    "raw": [c for c in tree_cols if c in raw_cols],
    "all_no_history": [c for c in tree_cols if c not in history_cols],
    "all": tree_cols,
    "all_reduced_v": [c for c in tree_cols if c not in v_drop],
}
print(f"Tree table: {len(tree_cols)} columns ({len(cat_cols)} categorical)")
for k, v in feature_sets.items():
    print(f"  feature set {k:15s} {len(v)} columns")

Tree table: 502 columns (34 categorical)
  feature set raw             430 columns
  feature set all_no_history  486 columns
  feature set all             502 columns
  feature set all_reduced_v   373 columns


## 10b. Target (mean) encoding and account behavior (from Jennifer)

Sections 10b to 10i come from Jennifer's notebook, which was built to drop in right here: after the tree table (Section 10) and before the neural-net table (Section 11). The last cell of this block appends the new columns to `tree`, `new_families` and `feature_sets`, so the neural-net table, checks, signal check and saved files pick them up automatically.

| Idea | Already in this notebook? | What this block adds |
|---|---|---|
| Target (mean) encoding | No; the advisor said to explore it | **10b**: smoothed, out-of-time encoding with a stability table |
| Is this device / email / address new for the account? | Partly (`prior_distinct_*` counts how many, not whether this one is new) | **10c**: `beh_*_new_*` flags |
| Transactions and amount in the previous 1 hour, 24 hours, 7 days and 30 days | No | **10d**: `vel_*` (the 7-day and 30-day windows come from Susie's notebook) |
| Amount minus the account's prior average | No (the ratio `*_amt_to_prior_mean` exists and is not repeated) | **10d** |
| Account amount spread: prior standard deviation, z-score, ratio to prior maximum | No | **10e** |
| Same exact amount repeated on the account | No | **10f** |

Every account feature is built for both keys (Key 1 and Key 2), so address-missing transactions are covered too. Column prefixes: `te_*` (target encoding), `beh_*` (behavior change), `vel_*` (velocity).

**Settings.** `JCONFIG` below holds this block's settings; it is saved with the outputs in Section 14 next to `CONFIG`.


In [15]:
JCONFIG = {
    # Target encoding: start with the columns found stable over time; test others (card1, card3, card5, addr1, id_31) one at a time
    "te_columns": ["card2", "P_emaildomain", "R_emaildomain"],
    "fpr_targets": [0.01, 0.05],        # for the TPR-at-fixed-FPR check
    "te_alpha": 20,                     # smoothing strength (see 10b)
    "te_folds": 5,                      # chronological folds for the training encoding
    "te_min_count_stability": 30,       # categories need this many rows in both halves to be compared

    # Behavioral and velocity features
    "windows": {"1h": 3_600, "24h": 86_400, "7d": 7 * 86_400, "30d": 30 * 86_400},   # 7d and 30d from Susie's config
    # account key -> {column checked for "is this value new for the account": short name}
    "new_value_specs": {
        "k1": {"device_family": "device", "P_emaildomain": "email"},
        "k2": {"device_family": "device", "addr1": "addr"},
    },
}

y_tr = train["isFraud"].to_numpy()


### 10b. Target encoding

**What it does.** Replaces each category with how often it was fraud in the past. A card, email domain, or browser that was often fraud in training gets a high number; a safe one gets a low number.

**Why here.** Section 6 uses *frequency* encoding (how common a category is, no labels). The advisor suggested mean encoding instead for many-category columns, but said it only works if each category's fraud rate is **stable over time**. This section builds it with a stability table, so the modeling pipeline can compare results with and without the `te_*` columns.

### The math

**Smoothed fraud rate.** For a column $c$ and category $k$, with $\text{fraud}(c,k)$ fraud cases among $n(c,k)$ training rows, $\bar y$ = the overall fraud rate, and smoothing strength $\alpha$:

$$\text{te}(c,k) = \frac{\text{fraud}(c,k) + \alpha\,\bar y}{n(c,k) + \alpha}$$

This pulls rare categories toward the overall rate. Example with $\alpha = 20$ and $\bar y = 0.035$: a card with 1 transaction that was fraud has a raw rate of 100%, but $\text{te} = (1 + 20 \times 0.035)/(1 + 20) = 0.081$, which is 8.1%. A card with 1,000 transactions and 50 frauds gets $(50 + 0.7)/1020 = 0.0497$, close to its raw 5%.

**Unseen and missing categories.** Missing values are their own category (as in the team's frequency encoding). A category never seen in the fitting data gets $\bar y$.

### Why training rows need a different encoding

If every training row were encoded with rates computed from *all* of training, each row's own label would be inside its own encoding. A model would learn to trust the encoding far too much, then fail on validation. So training rows use an **out-of-time** encoding:

1. Cut the training period into 5 chronological folds (boundaries moved to the start of a timestamp, like the team's split).
2. Encode fold $i$ using **only folds before it**.
3. Fold 0 has no past, so it gets a constant: fold 0's own overall fraud rate.

Validation, holdout, and Kaggle test are encoded with rates from **all of training**. They have no labels of their own in the maps, so there is no leakage.

**Trade-off.** Training rows see less history than validation rows do, so the encodings are slightly less sharp in training. That is the price of an honest training signal.

### Stability check

The table printed below compares each column's category fraud rates in the **first half** versus the **second half** of training (categories with enough rows in both halves only). A rank correlation near 1 means the rates are stable and mean encoding is safe; near 0 means they are not. The default list holds only the columns the EDA found stable (`card2`, `P_emaildomain`, `R_emaildomain`). `addr1` is left out because the team found it drifts; add it, `card1`, `card3`, `card5`, or `id_31` to `te_columns` one at a time and compare their rows in this table.

In [16]:
def fold_edges(t, n_folds):
    # Row positions where each chronological fold starts, moved to the first row of a timestamp
    t = np.asarray(t)
    edges = [0]
    for i in range(1, n_folds):
        row = min(int(len(t) * i / n_folds), len(t) - 1)
        edges.append(int(np.searchsorted(t, t[row], side="left")))
    edges.append(len(t))
    return edges

def te_fit(k, y, alpha):
    # Smoothed fraud rate per category. k = category labels (text), y = 0/1 labels.
    y = np.asarray(y, dtype="float64")
    g = pd.DataFrame({"k": np.asarray(k, dtype=object), "y": y}).groupby("k")["y"].agg(["sum", "count"])
    prior = float(y.mean())
    enc = (g["sum"] + prior * alpha) / (g["count"] + alpha)
    return enc, prior

def te_apply(k, enc, prior):
    mapped = pd.Series(np.asarray(k, dtype=object)).map(enc)
    return mapped.fillna(prior).astype("float32").to_numpy()

def te_oof_train(k, y, t, n_folds, alpha):
    # Training-period encoding: fold i is encoded using only the folds before it
    k = pd.Series(np.asarray(k, dtype=object))
    y = np.asarray(y)
    edges = fold_edges(t, n_folds)
    out = np.full(len(k), float(y[:edges[1]].mean()), dtype="float32")   # fold 0: constant
    for i in range(1, n_folds):
        past = slice(0, edges[i])
        cur = slice(edges[i], edges[i + 1])
        enc, prior = te_fit(k.iloc[past], y[past], alpha)
        out[cur] = te_apply(k.iloc[cur], enc, prior)
    return out

def auc_dir(y, x):
    # Direction-free AUC of a single column used as a score (same idea as the team's signal check)
    x = pd.Series(np.asarray(x, dtype="float64"))
    x = x.fillna(x.min() - 1 if x.notna().any() else 0)
    if x.nunique() < 2:
        return np.nan
    a = roc_auc_score(np.asarray(y), x)
    return max(a, 1 - a)

In [17]:
te_cols = [c for c in JCONFIG["te_columns"] if c in train.columns]
t_train = train["TransactionDT"].to_numpy()
edges = fold_edges(t_train, JCONFIG["te_folds"])
mid = fold_edges(t_train, 2)[1]

te_tables = {n: pd.DataFrame(index=pd.Index(f["TransactionID"], name="TransactionID")) for n, f in raw.items()}
te_rules = {}
for c in te_cols:
    ktr = cat_key(train[c]).to_numpy()
    te_tables["train"][f"te_{c}"] = te_oof_train(ktr, y_tr, t_train, JCONFIG["te_folds"], JCONFIG["te_alpha"])
    enc, prior = te_fit(ktr, y_tr, JCONFIG["te_alpha"])
    te_rules[c] = {"prior": prior, "alpha": JCONFIG["te_alpha"], "encoding": enc}
    for n in [p for p in raw if p != "train"]:
        te_tables[n][f"te_{c}"] = te_apply(cat_key(raw[n][c]).to_numpy(), enc, prior)

# Stability: do category fraud rates in the first half of training match the second half?
m = JCONFIG["te_min_count_stability"]
rows = []
for c in te_cols:
    k = cat_key(train[c]).to_numpy()
    a = pd.DataFrame({"k": k[:mid], "y": y_tr[:mid]}).groupby("k")["y"].agg(["mean", "count"])
    b = pd.DataFrame({"k": k[mid:], "y": y_tr[mid:]}).groupby("k")["y"].agg(["mean", "count"])
    both = a.join(b, lsuffix="_a", rsuffix="_b", how="inner")
    both = both[(both["count_a"] >= m) & (both["count_b"] >= m)]
    rho = both["mean_a"].corr(both["mean_b"], method="spearman") if len(both) > 2 else np.nan
    after_first = np.arange(len(train)) >= edges[1]
    rows.append({
        "column": c,
        "categories_in_train": int(len(a)),
        "categories_compared": int(len(both)),
        "rate_rank_corr_half1_vs_half2": rho,
        "train_auc_out_of_time": auc_dir(y_tr[after_first], te_tables["train"][f"te_{c}"].to_numpy()[after_first]),
        "validation_auc": auc_dir(labels["validation"], te_tables["validation"][f"te_{c}"]),
    })
te_report = pd.DataFrame(rows).set_index("column")
print(te_report.round(3).to_string())

               categories_in_train  categories_compared  rate_rank_corr_half1_vs_half2  train_auc_out_of_time  validation_auc
column                                                                                                                       
card2                          500                  395                          0.479                  0.710           0.728
P_emaildomain                   60                   51                          0.311                  0.587           0.589
R_emaildomain                   61                   30                          0.678                  0.677           0.683


## 10c. Behavioral-change features: is this value new for the account?

**Hypothesis.** A fraudster who takes over an existing account often brings something the account has never used: a new device, a new email, a new address.

**What Section 7 has.** `prior_distinct_*` says *how many* different devices or emails the account used before. It does not say whether **this** transaction's device or email is one of them. This section adds that.

### Definition

For account key $a$ and column $x$ (device family, email domain, or address), transaction $i$ gets:

$$\text{new}_i = \begin{cases} 1 & \text{if no earlier transaction of the account used the same } x \text{ value} \\ 0 & \text{if one did} \\ \text{missing} & \text{if the account has no earlier history, or } x \text{ or the key is missing} \end{cases}$$

"Earlier" means a **strictly earlier timestamp**, the team's rule. Two payments in the same second never see each other. Labels are never used.

**Why "no history" is missing, not 1.** An account's very first transaction has nothing to compare with, so every value would look new and the flag would just repeat "first transaction". The team's `has_prior` already says that. Here the flag only speaks when there is something to compare against.

**Worked example.** One account, all device families seen so far in order: T1 windows (t=10), T2 samsung (t=20), T3 samsung (t=20), T4 ios (t=30), T5 windows (t=50).

| | device | earlier devices | flag |
|---|---|---|---|
| T1 | windows | none (no history) | missing |
| T2 | samsung | windows | **1** (new) |
| T3 | samsung | windows (T2 is the same second, not earlier) | **1** (new) |
| T4 | ios | windows, samsung | **1** (new) |
| T5 | windows | windows, samsung, ios | **0** (seen before) |

### Which keys

| Key | Features | Why |
|---|---|---|
| Key 1 (card1, addr1, D1n) | new device, new email | the team's main key |
| Key 2 (card1, email domain, D1n) | new device, new address | covers address-missing transactions. Email is part of Key 2, so only device and address can be new. Address cannot be new inside Key 1 because `addr1` is part of that key |

### How the code does it quickly

For every (account, value) pair it finds the **first time** that pair appears. A transaction is "new" when its own time equals that first time, because then nothing earlier used the value. One grouped minimum replaces a search through each transaction's history.

In [18]:
def build_events():
    # Every transaction from every period with its account keys, indexed by TransactionID.
    # Later periods become history for even-later payments without labels, as in Section 7.
    cols = ["TransactionID", "TransactionDT", "TransactionAmt", "addr1", "device_family", "P_emaildomain"]
    parts = [f[cols].set_index("TransactionID").join(keys[n]) for n, f in raw.items()]
    ev = pd.concat(parts)
    assert ev.index.is_unique
    return ev

def new_value_features(ev, key, spec):
    # spec = {column: short name}. 1 = value never used before by this account, 0 = used before,
    # NaN = no key, value missing, or the account has no earlier history to compare with.
    elig = ev.loc[ev[key].notna()]
    acct_first = elig.groupby(key)["TransactionDT"].transform("min")
    has_prior = elig["TransactionDT"] > acct_first
    out = {}
    for col, short in spec.items():
        sub = elig.loc[elig[col].notna(), [key, col, "TransactionDT"]]
        first_seen = sub.groupby([key, col])["TransactionDT"].transform("min")
        is_new = (sub["TransactionDT"] == first_seen).to_numpy()
        usable = has_prior.loc[sub.index].to_numpy()
        s = pd.Series(np.nan, index=ev.index, dtype="float32")
        s.loc[sub.index[usable]] = is_new[usable].astype("float32")
        out[f"beh_{key}_new_{short}"] = s
    return pd.DataFrame(out)

## 10d. Velocity and amount deviation

### Velocity: how busy was the account just before this payment?

Fraud often arrives in bursts: several payments in a short time, or a large amount of money moving in a day. For each account key and each window $w$ (1 hour and 24 hours from Jennifer's notebook; 7 days and 30 days from Susie's, which catch slower build-ups over a week or a month):

$$\text{txn}_w(i) = \#\{\, j : \text{same account},\ t_i - w \le t_j < t_i \,\}, \qquad \text{amt}_w(i) = \sum_{j \text{ in that set}} \text{amount}_j$$

The window ends **strictly before** $t_i$, so the current payment and simultaneous payments are never counted.

**Worked example.** Account with payments at t = 1,000 ($50), 2,000 ($40), 3,000 ($200), and a new payment at t = 4,000. For the 1-hour window (3,600 seconds), all three earlier payments fall in $[400, 4000)$, so `txn_1h = 3` and `amt_1h = 290`.

**Missing vs. zero.** If the account key can be built but nothing happened in the window, the count is a true **0**. If the key cannot be built, the value is missing (the account is unknown).

### Amount deviation

$$\text{amt\_minus\_prior\_mean}_i = \text{amount}_i - \text{mean of all earlier amounts of the account}$$

Missing when there is no history. The team already has the **ratio** (`k1_amt_to_prior_mean`), so the ratio is not repeated here. The difference adds the size of the jump in dollars. Example: prior mean 87.50 and a 500 payment gives +412.50.

### How the code does it quickly

All transactions are placed on one sorted line, ordered by account and then time. For every transaction, two binary searches find where its window starts and where "strictly earlier" ends. A running total of amounts then gives the window's sum by subtraction, and the difference of the two positions gives the count. Each account sits in its own stretch of the line (`account_number × 2^26 + time`), so windows never reach into another account.

In [19]:
BIG = 1 << 26   # bigger than any TransactionDT + window, so each account has its own stretch of the sorted line

def window_features(ev, key, windows):
    mask = ev[key].notna().to_numpy()
    elig = ev.loc[mask, [key, "TransactionDT", "TransactionAmt"]]
    code = pd.factorize(elig[key])[0].astype("int64")
    t = elig["TransactionDT"].to_numpy().astype("int64")
    assert t.max() + max(windows.values()) < BIG, "BIG is too small for these times and windows"
    pos = code * BIG + t
    order = np.argsort(pos, kind="stable")
    S = pos[order]
    cs = np.concatenate([[0.0], np.cumsum(elig["TransactionAmt"].to_numpy("float64")[order])])
    hi = np.searchsorted(S, pos, side="left")                 # events strictly before this one
    start = np.searchsorted(S, code * BIG, side="left")       # first event of this account

    def spread(values):
        arr = np.full(len(ev), np.nan, dtype="float32")
        arr[mask] = np.asarray(values, dtype="float64")
        return arr

    out = {}
    for name, w in windows.items():
        lo = np.searchsorted(S, pos - w, side="left")
        out[f"vel_{key}_txn_{name}"] = spread(hi - lo)
        out[f"vel_{key}_amt_{name}"] = spread(cs[hi] - cs[lo])
    n_prior = hi - start
    prior_mean = np.where(n_prior > 0, (cs[hi] - cs[start]) / np.maximum(n_prior, 1), np.nan)
    amount = elig["TransactionAmt"].to_numpy("float64")
    out[f"beh_{key}_amt_minus_prior_mean"] = spread(amount - prior_mean)
    return pd.DataFrame(out, index=ev.index)

## 10e. Richer account amount statistics

**Idea.** `amt_minus_prior_mean` says how far a payment is from the account's average, but not whether that is *unusual for this account*. An account that always spends between $5 and $500 can absorb a $400 payment; an account that always spends $20 cannot. Adding the account's **spread** and **largest past amount** fixes that.

For each transaction, using only strictly earlier transactions of the same account ($n$ = how many, with amounts $a_j$):

$$\text{prior\_std} = \sqrt{\frac{\sum a_j^2 - (\sum a_j)^2 / n}{n - 1}} \quad (n \ge 2)$$

$$\text{amt\_z\_prior} = \frac{\text{amount} - \text{prior\_mean}}{\max(\text{prior\_std},\ 1)} \quad (n \ge 3)$$

$$\text{amt\_over\_prior\_max} = \frac{\text{amount}}{\max(\text{prior\_max},\ 1)}$$

- The z-score says how many "typical swings" away this amount is. It needs at least 3 earlier payments, because a spread from 1 or 2 points is not meaningful. The floor of 1 on the std (same idea as the team's floor on the mean) stops an account with nearly identical payments from producing an enormous z.
- The ratio to the prior max is above 1 exactly when this payment is **the largest the account has ever made**. That is a clean "new record" signal.

**Worked example.** An account's earlier payments are $10, $20, $40 and now a $80 payment arrives: mean 23.33, std 15.28, so z = (80 − 23.33) / 15.28 = **3.71**. The prior max is 40, so the ratio is **2.0** (double the account's previous record).

**Missing vs. zero.** No history, or too little history for the statistic, means missing, never 0 (0 would claim "spread of zero").

**How the code does it.** The std reuses the sorted-line trick from Section 10d with a second running total, the sum of squared amounts, so variance comes from subtraction. The prior max uses a running maximum per account over *timestamp groups*, shifted down by one group, so simultaneous payments never see each other. The team's median would need a different method, so it is left out.

In [20]:
def amount_history_features(ev, key, min_prior=3):
    # Prior std, z-score, and ratio to the prior maximum, from strictly earlier payments of the same account
    mask = ev[key].notna().to_numpy()
    elig = ev.loc[mask, [key, "TransactionDT", "TransactionAmt"]]
    code = pd.factorize(elig[key])[0].astype("int64")
    t = elig["TransactionDT"].to_numpy().astype("int64")
    assert t.max() < BIG, "BIG is too small for these times"
    amount = elig["TransactionAmt"].to_numpy("float64")
    pos = code * BIG + t
    order = np.argsort(pos, kind="stable")
    S = pos[order]
    cs = np.concatenate([[0.0], np.cumsum(amount[order])])
    cs2 = np.concatenate([[0.0], np.cumsum(amount[order] ** 2)])
    hi = np.searchsorted(S, pos, side="left")                 # events strictly before this one
    start = np.searchsorted(S, code * BIG, side="left")       # first event of this account
    n = hi - start
    s1 = cs[hi] - cs[start]
    s2 = cs2[hi] - cs2[start]
    mean = np.where(n > 0, s1 / np.maximum(n, 1), np.nan)
    var = np.where(n >= 2, (s2 - s1 ** 2 / np.maximum(n, 1)) / np.maximum(n - 1, 1), np.nan)
    std = np.sqrt(np.clip(var, 0, None))                      # missing stays missing
    z = np.where(n >= min_prior, (amount - mean) / np.maximum(std, 1.0), np.nan)

    # Prior max: running maximum over timestamp groups, shifted by one group (strictly earlier)
    bt = elig.groupby([key, "TransactionDT"], sort=True)["TransactionAmt"].max().rename("mx").reset_index()
    bt["cm"] = bt.groupby(key, sort=False)["mx"].cummax()
    bt["prior_max"] = bt.groupby(key, sort=False)["cm"].shift(1)
    pm = (elig[[key, "TransactionDT"]]
          .join(bt.set_index([key, "TransactionDT"])["prior_max"], on=[key, "TransactionDT"])["prior_max"]
          .to_numpy("float64"))
    ratio = amount / np.maximum(pm, 1.0)

    def spread(values):
        arr = np.full(len(ev), np.nan, dtype="float32")
        arr[mask] = values
        return arr

    return pd.DataFrame({
        f"beh_{key}_prior_std_amt": spread(std),
        f"beh_{key}_amt_z_prior": spread(z),
        f"beh_{key}_amt_over_prior_max": spread(ratio),
    }, index=ev.index)

## 10f. Repeated amounts on the account

**Idea.** Two opposite behaviors both show up as a repeated amount. Legitimate accounts repeat amounts (a subscription, a usual coffee order). Fraudsters often **test a stolen card** with the same small amount several times, or run the same amount again and again. Either way, "has this account paid exactly this amount before, and how recently?" is more informative than the amount alone. The model learns which direction matters.

For each transaction, using strictly earlier payments of the same account **with the same amount to the cent**:

$$\text{prior\_same\_amt\_count} = \#\{\, j : \text{same account},\ \text{same amount},\ t_j < t_i \,\}$$

$$\text{secs\_since\_same\_amt} = t_i - \text{time of the most recent such payment}$$

**Worked example.** One account pays $10 at t=10, $20 at t=20, $20 at t=20 (same second), $10 at t=30, $10 at t=50.

| Payment | Earlier same amount | count | secs since |
|---|---|---|---|
| $10 at t=10 | none | 0 | missing |
| $20 at t=20 | none (the other $20 is the same second) | 0 | missing |
| $20 at t=20 (second one) | none (same reason) | 0 | missing |
| $10 at t=30 | t=10 | **1** | **20** |
| $10 at t=50 | t=10 and t=30 | **2** | **20** |

**Missing vs. zero.** If the account key exists, a count of 0 is a true 0. If the key cannot be built, both values are missing. `secs_since_same_amt` is missing whenever the amount is new for the account.

**How the code does it.** Same pattern as the team's history features: group by account, amount, and exact timestamp; take a running count within each (account, amount) pair and subtract the current moment so simultaneous payments stay hidden.

In [21]:
def repeat_amount_features(ev, key):
    # How often, and how recently, this account paid exactly this amount before (strictly earlier payments)
    elig = ev.loc[ev[key].notna(), [key, "TransactionDT", "TransactionAmt"]].copy()
    elig["cents"] = np.rint(elig["TransactionAmt"].to_numpy("float64") * 100).astype("int64")
    bt = (elig.groupby([key, "cents", "TransactionDT"], sort=True).size()
              .rename("n").reset_index())
    g = bt.groupby([key, "cents"], sort=False)
    bt["prior_same"] = g["n"].cumsum() - bt["n"]
    bt["secs_same"] = bt["TransactionDT"] - g["TransactionDT"].shift(1)
    joined = elig.join(bt.set_index([key, "cents", "TransactionDT"])[["prior_same", "secs_same"]],
                       on=[key, "cents", "TransactionDT"])
    out = pd.DataFrame({
        f"beh_{key}_prior_same_amt_count": joined["prior_same"].astype("float32"),
        f"beh_{key}_secs_since_same_amt": joined["secs_same"].astype("float32"),
    })
    return out.reindex(ev.index)

In [22]:
ev = build_events()
beh_parts = [new_value_features(ev, key, spec) for key, spec in JCONFIG["new_value_specs"].items()]
beh_parts += [window_features(ev, key, JCONFIG["windows"]) for key in JCONFIG["new_value_specs"]]
beh_parts += [amount_history_features(ev, key) for key in JCONFIG["new_value_specs"]]
beh_parts += [repeat_amount_features(ev, key) for key in JCONFIG["new_value_specs"]]
behavior_all = pd.concat(beh_parts, axis=1)
del ev, beh_parts
gc.collect()

beh_tables = {}
for n, f in raw.items():
    ids = f["TransactionID"].to_numpy()
    t_ = behavior_all.loc[ids]
    t_.index = pd.Index(ids, name="TransactionID")
    beh_tables[n] = t_

print(f"{behavior_all.shape[1]} behavior / velocity columns")
print((behavior_all.notna().mean().round(3)).rename("share_not_missing").to_string())

32 behavior / velocity columns
beh_k1_new_device              0.053
beh_k1_new_email               0.450
beh_k2_new_device              0.100
beh_k2_new_addr                0.458
vel_k1_txn_1h                  0.875
vel_k1_amt_1h                  0.875
vel_k1_txn_24h                 0.875
vel_k1_amt_24h                 0.875
vel_k1_txn_7d                  0.875
vel_k1_amt_7d                  0.875
vel_k1_txn_30d                 0.875
vel_k1_amt_30d                 0.875
beh_k1_amt_minus_prior_mean    0.556
vel_k2_txn_1h                  0.845
vel_k2_amt_1h                  0.845
vel_k2_txn_24h                 0.845
vel_k2_amt_24h                 0.845
vel_k2_txn_7d                  0.845
vel_k2_amt_7d                  0.845
vel_k2_txn_30d                 0.845
vel_k2_amt_30d                 0.845
beh_k2_amt_minus_prior_mean    0.534
beh_k1_prior_std_amt           0.422
beh_k1_amt_z_prior             0.341
beh_k1_amt_over_prior_max      0.556
beh_k2_prior_std_amt           0.402
beh_k2_

## 10g. Assemble one table per period

Target-encoding and behavior columns are placed side by side, one row per transaction in the same order as the tree tables. The index is `TransactionID`, so joining to the tree table is a plain column-wise concatenation.

In [23]:
jen = {n: pd.concat([te_tables[n], beh_tables[n]], axis=1) for n in raw}
jen_cols = jen["train"].columns.tolist()

jen_families = {
    "target_encoding": [f"te_{c}" for c in te_cols],
    "behavior_new_value": [c for c in jen_cols if c.startswith("beh_") and "_new_" in c],
    "amount_deviation": [c for c in jen_cols
                         if c.endswith(("amt_minus_prior_mean", "prior_std_amt", "amt_z_prior", "amt_over_prior_max"))],
    "repeat_amount": [c for c in jen_cols if "same_amt" in c],
    "velocity": [c for c in jen_cols if c.startswith("vel_")],
}
assert sorted(sum(jen_families.values(), [])) == sorted(jen_cols)
for fam, cols in jen_families.items():
    print(f"{fam:20s} {len(cols):>2d}  {cols}")

target_encoding       3  ['te_card2', 'te_P_emaildomain', 'te_R_emaildomain']
behavior_new_value    4  ['beh_k1_new_device', 'beh_k1_new_email', 'beh_k2_new_device', 'beh_k2_new_addr']
amount_deviation      8  ['beh_k1_amt_minus_prior_mean', 'beh_k2_amt_minus_prior_mean', 'beh_k1_prior_std_amt', 'beh_k1_amt_z_prior', 'beh_k1_amt_over_prior_max', 'beh_k2_prior_std_amt', 'beh_k2_amt_z_prior', 'beh_k2_amt_over_prior_max']
repeat_amount         4  ['beh_k1_prior_same_amt_count', 'beh_k1_secs_since_same_amt', 'beh_k2_prior_same_amt_count', 'beh_k2_secs_since_same_amt']
velocity             16  ['vel_k1_txn_1h', 'vel_k1_amt_1h', 'vel_k1_txn_24h', 'vel_k1_amt_24h', 'vel_k1_txn_7d', 'vel_k1_amt_7d', 'vel_k1_txn_30d', 'vel_k1_amt_30d', 'vel_k2_txn_1h', 'vel_k2_amt_1h', 'vel_k2_txn_24h', 'vel_k2_amt_24h', 'vel_k2_txn_7d', 'vel_k2_amt_7d', 'vel_k2_txn_30d', 'vel_k2_amt_30d']


## 10h. Checks and signal check for this block

**Unit tests on a toy account.** These confirm that simultaneous payments cannot see each other, that a future payment changes nothing earlier, and that row order does not matter. They use made-up data, so they run even without the Kaggle files.

**Pipeline checks.** Every period has the same columns and aligns with the team's rows; encodings have no missing values; and a training label's flip changes **no** encoding that should not depend on it.

**Signal check.** Same idea as Section 13: each new column used alone as a fraud score, direction-free AUC on training and validation, plus TPR at a fixed FPR (what share of fraud is caught if only 1% or 5% of legitimate transactions may be flagged) for the target-encoded columns. Training AUC skips the first target-encoding fold, which is a constant. Use it to understand the features, not to pick them; picking belongs in the modeling pipeline.

In [24]:
# ---- unit tests on a toy account (no data needed)
toy = pd.DataFrame({
    "TransactionDT": [10, 20, 20, 30, 40, 50],
    "TransactionAmt": [10.0, 20.0, 40.0, 80.0, 5.0, 15.0],
    "acct": ["A", "A", "A", "A", None, "A"],
    "dev": ["x", "y", "y", "z", "x", "x"],
}, index=pd.Index([1, 2, 3, 4, 5, 6], name="TransactionID"))
toy["acct"] = toy["acct"].astype("string")

nv = new_value_features(toy, "acct", {"dev": "dev"})["beh_acct_new_dev"]
# row 1 no history; rows 2 and 3 are the same second so neither sees the other; row 4 new; row 5 no key; row 6 seen at t=10
np.testing.assert_array_equal(nv.to_numpy(), np.array([np.nan, 1, 1, 1, np.nan, 0], dtype="float32"))

wf = window_features(toy, "acct", {"w": 15})
np.testing.assert_allclose(wf["vel_acct_txn_w"], [0, 1, 1, 2, np.nan, 0], equal_nan=True)
np.testing.assert_allclose(wf["vel_acct_amt_w"], [0, 10, 10, 60, np.nan, 0], equal_nan=True)
np.testing.assert_allclose(wf["beh_acct_amt_minus_prior_mean"],
                           [np.nan, 10, 30, 80 - 70 / 3, np.nan, 15 - 37.5], rtol=1e-5, equal_nan=True)

future = pd.DataFrame({"TransactionDT": [99], "TransactionAmt": [1.0], "acct": ["A"], "dev": ["w"]},
                      index=pd.Index([7], name="TransactionID"))
future["acct"] = future["acct"].astype("string")
bigger = pd.concat([toy, future])
pd.testing.assert_frame_equal(wf, window_features(bigger, "acct", {"w": 15}).loc[toy.index])
pd.testing.assert_frame_equal(new_value_features(toy, "acct", {"dev": "dev"}),
                              new_value_features(bigger, "acct", {"dev": "dev"}).loc[toy.index])
shuffled = toy.sample(frac=1, random_state=1)
pd.testing.assert_frame_equal(wf, window_features(shuffled, "acct", {"w": 15}).loc[toy.index])

# target encoding on a toy column
enc, prior = te_fit(np.array(["a", "a", "b"], dtype=object), [1, 0, 0], alpha=2)
assert np.isclose(prior, 1 / 3)
assert np.isclose(enc["a"], (1 + 2 / 3) / 4) and np.isclose(enc["b"], (0 + 2 / 3) / 3)
assert np.isclose(te_apply(np.array(["a", "__never_seen__"], dtype=object), enc, prior)[1], prior)
# amount history (Section 5b): prior std, z-score (needs 3+ priors), ratio to prior max
ah = amount_history_features(toy, "acct")
np.testing.assert_allclose(ah["beh_acct_prior_std_amt"],
                           [np.nan, np.nan, np.nan, np.sqrt(700 / 3), np.nan, np.sqrt(2875 / 3)], rtol=1e-4, equal_nan=True)
np.testing.assert_allclose(ah["beh_acct_amt_z_prior"],
                           [np.nan, np.nan, np.nan, (80 - 70 / 3) / np.sqrt(700 / 3), np.nan,
                            (15 - 37.5) / np.sqrt(2875 / 3)], rtol=1e-4, equal_nan=True)
np.testing.assert_allclose(ah["beh_acct_amt_over_prior_max"],
                           [np.nan, 2.0, 4.0, 2.0, np.nan, 0.1875], rtol=1e-4, equal_nan=True)
pd.testing.assert_frame_equal(ah, amount_history_features(bigger, "acct").loc[toy.index])
pd.testing.assert_frame_equal(ah, amount_history_features(shuffled, "acct").loc[toy.index])

# repeated amounts (Section 5c)
toy2 = toy.copy()
toy2["TransactionAmt"] = [10.0, 20.0, 20.0, 10.0, 5.0, 10.0]
ra = repeat_amount_features(toy2, "acct")
np.testing.assert_allclose(ra["beh_acct_prior_same_amt_count"], [0, 0, 0, 1, np.nan, 2], equal_nan=True)
np.testing.assert_allclose(ra["beh_acct_secs_since_same_amt"], [np.nan, np.nan, np.nan, 20, np.nan, 20], equal_nan=True)
pd.testing.assert_frame_equal(ra, repeat_amount_features(toy2.sample(frac=1, random_state=3), "acct").loc[toy2.index])
print("Unit tests passed.")

Unit tests passed.


In [25]:
# ---- pipeline checks
te_names = jen_families["target_encoding"]
for n in jen:
    assert jen[n].columns.tolist() == jen_cols
    assert jen[n].index.is_unique and jen[n].index.equals(pd.Index(raw[n]["TransactionID"]))
    assert not jen[n][te_names].isna().any().any(), f"missing target encodings in {n}"
if "tree_cols" in globals():
    assert set(jen_cols).isdisjoint(tree_cols), "column name clash with the team's tree table"

# A label in fold 1 must not change any encoding in folds 0 and 1, but must change later ones
c = te_cols[0]
kc = cat_key(train[c]).to_numpy()
base = te_oof_train(kc, y_tr, t_train, JCONFIG["te_folds"], JCONFIG["te_alpha"])
y_flip = y_tr.copy()
y_flip[edges[1] + 5] ^= 1
pert = te_oof_train(kc, y_flip, t_train, JCONFIG["te_folds"], JCONFIG["te_alpha"])
assert np.array_equal(base[:edges[2]], pert[:edges[2]])
assert not np.array_equal(base[edges[2]:], pert[edges[2]:])
print("All pipeline checks passed.")

All pipeline checks passed.


In [26]:
# ---- signal check
after_first = np.arange(len(train)) >= edges[1]
fam_of = {c: fam for fam, cols in jen_families.items() for c in cols}
signal_j = pd.DataFrame({
    "family": [fam_of[c] for c in jen_cols],
    "train_auc": [auc_dir(y_tr[after_first], jen["train"][c].to_numpy()[after_first]) for c in jen_cols],
    "validation_auc": [auc_dir(labels["validation"], jen["validation"][c]) for c in jen_cols],
}, index=jen_cols)
print(signal_j.sort_values("validation_auc", ascending=False).round(3).to_string())

# TPR at a fixed FPR, the advisor-style metric, for each single target-encoded column on validation
from sklearn.metrics import roc_curve

def tpr_at_fpr(y, score, targets):
    s = pd.Series(np.asarray(score, dtype="float64"))
    s = s.fillna(s.min() - 1 if s.notna().any() else 0)
    if roc_auc_score(np.asarray(y), s) < 0.5:
        s = -s                                   # use the direction that separates fraud best
    fpr, tpr, _ = roc_curve(np.asarray(y), s)
    return {f"tpr_at_{int(t * 100)}pct_fpr": float(np.interp(t, fpr, tpr)) for t in targets}

tpr_rows = {c: tpr_at_fpr(labels["validation"], jen["validation"][c], JCONFIG["fpr_targets"]) for c in te_names}
print("\nValidation TPR at fixed FPR, single target-encoded column (baseline, not a model):")
print(pd.DataFrame(tpr_rows).T.round(3).to_string())

                                         family  train_auc  validation_auc
te_card2                        target_encoding      0.710           0.728
te_R_emaildomain                target_encoding      0.677           0.683
beh_k1_prior_same_amt_count       repeat_amount      0.605           0.612
vel_k1_txn_1h                          velocity      0.595           0.601
vel_k1_amt_1h                          velocity      0.594           0.600
te_P_emaildomain                target_encoding      0.587           0.589
beh_k1_amt_over_prior_max      amount_deviation      0.569           0.585
beh_k1_amt_minus_prior_mean    amount_deviation      0.567           0.582
beh_k1_new_email             behavior_new_value      0.566           0.581
beh_k2_new_addr              behavior_new_value      0.560           0.579
vel_k1_txn_30d                         velocity      0.586           0.576
vel_k2_txn_7d                          velocity      0.598           0.572
vel_k1_txn_24h           

## 10i. Merge into the tree tables

Appends Jennifer's columns to every period's `tree` table, registers the new feature families, and updates the named feature sets: `all` now includes everything, `all_no_jennifer` leaves this block out (for the with/without comparison), and `all_no_history` also leaves out the `beh_*` and `vel_*` columns, since they are account history too. The target encodings stay in `all_no_history`, because they are not account history. Running it twice is harmless.

In [27]:
if set(jen_cols) <= set(tree["train"].columns):
    print("Already merged.")
else:
    for n in tree:
        assert tree[n].index.equals(jen[n].index), f"row order differs in {n}"
        tree[n] = pd.concat([tree[n], jen[n]], axis=1)
    tree_cols = tree["train"].columns.tolist()
    new_families.update(jen_families)
    history_cols = history_cols + [c for c in jen_cols if c.startswith(("beh_", "vel_"))]
    feature_sets["all"] = tree_cols
    feature_sets["all_no_jennifer"] = [c for c in tree_cols if c not in jen_cols]
    feature_sets["all_no_history"] = [c for c in tree_cols if c not in history_cols]
    feature_sets["all_reduced_v"] = [c for c in tree_cols if c not in v_drop]
    print(f"Merged: tree table now has {len(tree_cols)} columns")
    for k, v in feature_sets.items():
        print(f"  feature set {k:18s} {len(v)} columns")


Merged: tree table now has 537 columns
  feature set raw                430 columns
  feature set all_no_history     489 columns
  feature set all                537 columns
  feature set all_reduced_v      408 columns
  feature set all_no_jennifer    502 columns


## 11. Neural-net table

**What this cell does.** It builds the **input table** for the neural net. No model is trained here. It starts from the tree table (Section 10) and reshapes it into the form a neural net needs.

**Why a separate table.** The advisor's point: a tree splits on values and doesn't care about scale or outliers, but a neural net does math with the actual values. It needs every input to be a number, nothing missing, and everything on a similar scale. A net learns poorly when one input runs from 0 to 30,000 and another from 0 to 1.

Every rule below is **learned from training only**, then applied unchanged to validation, holdout, and Kaggle test.

### Which columns go in

| Piece | Source | Handling |
|---|---|---|
| Number columns (amounts, C and D columns, `freq_*`, account history, basic features) | Sections 6, 7, 10 | scaled (steps below) |
| V columns | Section 9 | only the 210 kept columns; the 129 near-duplicates are left out |
| Few-category text columns (`ProductCD`, `card4`, `card6`, `DeviceType`, `M4`, ...) | Section 4 | one-hot encoded |
| Missing-value flags | Section 8 | one per block, plus one per single column |

Left out: raw many-category codes like `card1` and `addr1` (their `freq_*` columns stand in) and many-category text columns like `P_emaildomain` (same).

### Step 1: Log-transform heavily skewed columns

On a 100,000-row training sample, a column is marked skewed if it is never negative and has a long tail of large values (skew above 2). Transaction amounts are a typical example. Those columns get

$$x \rightarrow \log(1 + x)$$

which squeezes the long tail so a few huge values don't dominate.

### Step 2: Clip extremes

For each column, values below the training 0.5th percentile or above the 99.5th percentile are pulled in to those limits. This keeps rare outliers from distorting training.

### Step 3: Fill missing values

Blanks are filled with the column's training median. Before filling, the missing-value flags are built from the original data (Section 8), because once a blank is filled it looks like a normal number. The flag is the only record that it was missing, and the EDA found missingness carries fraud signal.

Jennifer's account-history columns (`beh_*`, Section 10c to 10f) are missing when an account has no earlier history; they are filled the same way, and `k1_has_prior` / `k2_has_prior` keep the record of which transactions had no history.

### Step 4: Standardize

Each column is centered and scaled using its training mean and standard deviation:

$$z = \frac{x - \text{mean}}{\text{standard deviation}}$$

After this, every number column is centered around 0, with most values between about −3 and 3. If a column never varies (standard deviation 0), 1 is used to avoid dividing by zero.

**Example.** A \$500 transaction: $\log(1 + 500) \approx 6.22$. If the training mean of the logged amounts were 4.2 with a standard deviation of 1.0, the final value is $(6.22 - 4.2) / 1.0 \approx 2.0$: about two standard deviations above a typical amount.

### Step 5: One-hot encode the few-category columns

Each few-category column becomes one 0/1 column per category, using the fixed category lists from Section 10. For example, `card6` becomes `card6_credit`, `card6_debit`, `card6___MISSING__`, and `card6___UNSEEN__`. Because the lists come from training, every period gets exactly the same one-hot columns.

### Checks

The cell stops with an error unless every period has exactly the same columns in the same order, and the training table has no missing values left. It then prints the column counts and how many columns got the log transform.

**This is a starting point.** Gene/Kyaw is researching neural-net feature requirements, so these rules should be updated with whatever that research finds (for example, learned embeddings instead of frequency encoding for the many-category columns).

In [28]:
if CONFIG["save_nn_track"]:
    onehot_cols = [c for c in cat_cols if len(vocab[c]) - 2 <= CONFIG["onehot_max_levels"]]
    nn_num = [c for c in tree_cols
              if c not in cat_cols and c not in v_drop and c not in high_card and not c.startswith("na_block_")]
    na_groups = {**{name: cols[0] for name, cols in block_names.items()},
                 **{f"na_{c}": c for c in singles if c in nn_num}}

    t = tree["train"][nn_num].astype("float32")
    sample = t.sample(min(100_000, len(t)), random_state=0)
    log_cols = [c for c in nn_num if sample[c].min() >= 0 and sample[c].skew() > 2]
    t_log = t.copy()
    t_log[log_cols] = np.log1p(t_log[log_cols].clip(lower=0))
    lo, hi = t_log.quantile(0.005), t_log.quantile(0.995)
    clipped = t_log.clip(lo, hi, axis=1)
    med = clipped.median().fillna(0)          # a column with no training values at all gets 0
    filled = clipped.fillna(med)
    mu, sd = filled.mean().fillna(0), filled.std().replace(0, 1).fillna(1)
    del t, t_log, clipped, filled, sample
    gc.collect()

    def nn_table(n):
        X = tree[n]
        num = X[nn_num].astype("float32")
        num[log_cols] = np.log1p(num[log_cols].clip(lower=0))
        num = ((num.clip(lo, hi, axis=1).fillna(med) - mu) / sd).astype("float32")
        src = raw[n]  # same row order as tree[n]
        flags = pd.DataFrame({name: src[col].isna().astype("float32").to_numpy()
                              for name, col in na_groups.items()}, index=X.index)
        dummies = pd.get_dummies(X[onehot_cols], prefix=onehot_cols, dtype="float32")
        return pd.concat([num, flags, dummies], axis=1)

    nn = {n: nn_table(n) for n in tree}
    nn_cols = nn["train"].columns.tolist()
    assert all(nn[n].columns.tolist() == nn_cols for n in nn)
    assert not nn["train"].isna().any().any()
    print(f"Neural-net table: {len(nn_cols)} columns "
          f"({len(nn_num)} numeric, {len(na_groups)} missing flags, one-hot from {onehot_cols}); "
          f"log1p applied to {len(log_cols)} columns")

Neural-net table: 515 columns (343 numeric, 48 missing flags, one-hot from ['ProductCD', 'card4', 'card6', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'email_domain_relationship', 'na_cluster']); log1p applied to 246 columns


## 12. Checks

These confirm the rules protect against leakage and that every table lines up. They do not show that the features improve fraud detection; that's the modeling pipeline's job.

In [29]:
# 1. Tables line up across periods, and no forbidden inputs slipped in
forbidden = {"isFraud", "TransactionID", "TransactionDT", "DeviceInfo", "D1n", "k1", "k2"}
assert forbidden.isdisjoint(tree_cols)
for n in tree:
    assert tree[n].columns.tolist() == tree_cols and tree[n].index.is_unique
    assert tree[n].index.equals(meta[n].index)
    if n in labels:
        assert tree[n].index.equals(labels[n].index)
    for c in cat_cols:
        assert tree[n][c].cat.categories.tolist() == vocab[c]

# 2. History uses strictly earlier timestamps; simultaneous payments can't see each other;
#    distinct counts only count values first seen before; a future payment changes nothing earlier
toy = pd.DataFrame({
    "TransactionDT": [10, 20, 20, 30, 40],
    "TransactionAmt": [10.0, 20.0, 40.0, 80.0, 5.0],
    "acct": ["A", "A", "A", "A", pd.NA],
    "dev": ["x", "y", "x", "z", "x"],
}, index=pd.Index([1, 2, 3, 4, 5], name="TransactionID"))
th = strict_past(toy, "acct", ["dev"], "t")
assert th["t_prior_count"].tolist() == [0, 1, 1, 3, 0]
assert th.loc[[2, 3], "t_prior_mean_amt"].eq(10).all()
assert np.isclose(th.loc[4, "t_prior_mean_amt"], 70 / 3)
assert th["t_prior_distinct_dev"].tolist() == [0, 1, 1, 2, 0]
future = pd.DataFrame({"TransactionDT": [99], "TransactionAmt": [1.0], "acct": ["A"], "dev": ["w"]}, index=[6])
pd.testing.assert_frame_equal(th, strict_past(pd.concat([toy, future]), "acct", ["dev"], "t").loc[toy.index])
pd.testing.assert_frame_equal(th, strict_past(toy.sample(frac=1, random_state=1), "acct", ["dev"], "t").loc[toy.index])

# 3. Unseen categories get frequency 0
probe = raw["validation"].iloc[:1].copy()
for c in high_card:
    probe[c] = "__never_seen__" if not pd.api.types.is_numeric_dtype(probe[c]) else -12345.0
assert frequency_encode(probe).eq(0).all().all()

# 4. Amount features: unseen product uses the overall median; unseen amount gets frequency 0
probe = raw["validation"].iloc[:1].copy()
probe["ProductCD"] = "__never_seen__"
probe["TransactionAmt"] = cents_key(pd.Series(amount_freq_map.index)).max() / 100 + 1.23
a = amount_time_features(probe).iloc[0]
assert np.isclose(a["amount_to_product_median"], probe["TransactionAmt"].iloc[0] / max(global_amount_median, 1))
assert a["amount_train_frequency"] == 0

# 5. Every period uses the same missingness clusters
assert all(tree[n]["na_cluster"].cat.categories.tolist() == vocab["na_cluster"] for n in tree)
print("All checks passed.")

All checks passed.


## 13. Quick signal check and the advisor's KS (Kolmogorov-Smirnov) table

This section has two tools built on the same idea: **pick a cutoff, call everything above it "fraud," and count what you caught and what you wrongly flagged.**

- The **signal check** uses that idea to score each new feature on its own, before any model exists.
- The **KS table** uses the same idea to judge a model's fraud scores in the format the advisor asked for. It's built here, and used for real once the modeling pipeline produces predictions.

### Signal check

**What it does.** For each new feature, it measures how well that one feature alone separates fraud from legitimate transactions, on training and on validation. No model is trained. The feature's own values are used as the score, the way a model's predicted fraud probability normally would be.

**Why no model is needed.** A confusion matrix only needs a yes/no prediction and the true label for each transaction. The predictions can come from anything: a model, a business rule, or a single feature with a cutoff. An ROC curve only needs a number to sweep cutoffs over.

#### How it works

1. **Treat the feature as a score.** Missing values are set just below the column's smallest value, so they rank lowest.
2. **Sweep every cutoff.** At each cutoff, call everything at or above it "fraud." That gives one confusion matrix, and from it the true positive rate (share of fraud caught) and false positive rate (share of legitimate transactions flagged).
3. **Draw the ROC curve.** Plot each cutoff's false positive rate (x-axis) against its true positive rate (y-axis).
4. **Take the area under it (AUC).** 0.5 means no better than a coin flip; 1.0 means perfect separation.

**An equivalent way to read AUC.** It is the probability that a randomly chosen fraud transaction has a higher value than a randomly chosen legitimate one.

**Direction-free.** Some features work in reverse (lower values mean more fraud), which gives an AUC below 0.5. The code reports $\max(\text{AUC},\ 1 - \text{AUC})$, so the number measures how much signal there is, not its direction.

**Constant columns.** If a feature has only one value in a period, it can't rank anything, so the result is blank (NaN).

#### Example

Six transactions scored by one feature: A = 0.9 (fraud), B = 0.8, C = 0.6 (fraud), D = 0.4, E = 0.3, F = 0.2.

There are 2 × 4 = 8 (fraud, legit) pairs. Fraud A beats all 4 legitimate transactions; fraud C beats D, E, and F but not B. That's 7 of 8, so AUC = **0.875**. Sweeping the cutoffs and measuring the area under the ROC curve gives the same 0.875.

**Check with real numbers.** For a 0/1 feature there is only one cutoff, so

$$\text{AUC} = \frac{1 + \text{TPR} - \text{FPR}}{2}$$

About 54.8% of training fraud has identity data (TPR) and about 24.4% of legitimate transactions do (FPR), giving $(1 + 0.548 - 0.244)/2 = 0.652$. This matches the training AUC for `has_identity`.

#### What the full run showed

These notes come from the run before Jennifer's features (Section 10b) were added; the numbers can shift if features or settings change. In Jennifer's own run on the same split, `te_card2` scored **0.728** on validation, higher than any feature below, so it should now top the table; `te_R_emaildomain` scored 0.683.

- **The strongest features are mostly one signal repeated.** `raw_missing_count` (0.678 on validation), `freq_id_31`, `has_identity`, `identity_missing_count`, and several `na_block_*` flags all score about 0.66 to 0.68. They all reflect whether a transaction has identity data (7.55% fraud with it vs. 2.13% without, per the EDA): transactions with identity data have fewer blanks.
- **Address-missing shows up as expected.** `freq_addr1` (0.652) and `k1_key_missing` (0.637) capture the EDA's highest-risk group.
- **The best history feature is `k1_secs_since_prev`** (0.643 on validation).
- **Key 2 features score low overall** (about 0.50 to 0.58), but Key 2 was built for the address-missing group, which is only 11% of transactions. Scoring it across all transactions dilutes its effect; it should also be checked within that group.
- **Some blocks are almost never missing.** `na_block_01`, `02`, and `12` score about 0.50 and are candidates to drop.
- **Training and validation mostly agree**, a sign the signals hold up over time. Exceptions: `k2_prior_count` and `k2_amt_to_prior_mean` fall from about 0.55 to 0.51.

#### Limits

- **AUC only rewards steady trends** ("higher means more fraud"). It misses bumps, like the EDA's fraud peak in relative hours 5 to 10, which is why `hour_cos` scores only 0.509. Tree models can use bumps that AUC misses.
- **Features can help in combination.** Amount means more when the model also knows the product. A low score here doesn't make a feature useless.
- **Use this to understand features, not to pick them.** Dropping features because they scored low on validation would make validation less honest. Feature choices should come from the modeling pipeline's comparisons, and the final number from the holdout.
- **It sets a baseline.** Once a real model exists, it should clearly beat the best single feature (about 0.73 with target encoding). If it doesn't, something is wrong.

### From the ROC curve to the KS table

The ROC curve sweeps **every** cutoff and summarizes them in one number (AUC). A fraud team reviewing alerts needs something more concrete: if we review the top 10% most suspicious transactions, how much fraud do we catch, and how many good customers do we bother? The KS table answers that by picking about 10 cutoffs and showing each one as a row. Each row is one confusion matrix, summarized.

#### Example

Six transactions, scored by some feature:

| Transaction | Feature value | Actually fraud? |
|---|---|---|
| A | 0.9 | yes |
| B | 0.8 | no |
| C | 0.6 | yes |
| D | 0.4 | no |
| E | 0.3 | no |
| F | 0.2 | no |

2 fraud, 4 legit.

**There is no single cutoff.** AUC tries every cutoff: each distinct feature value is used once. (0.5 is a common default for turning a model's probability into a final yes/no decision, but AUC never needs that decision.) Sweeping the cutoff from high to low:

| Flag if value ≥ | Fraud caught (TPR) | Legit flagged (FPR) |
|---|---|---|
| above 0.9 (flag nothing) | 0 of 2 = 0.00 | 0 of 4 = 0.00 |
| 0.9 | 1 of 2 = 0.50 | 0 of 4 = 0.00 |
| 0.8 | 1 of 2 = 0.50 | 1 of 4 = 0.25 |
| 0.6 | 2 of 2 = 1.00 | 1 of 4 = 0.25 |
| 0.4 | 2 of 2 = 1.00 | 2 of 4 = 0.50 |
| 0.3 | 2 of 2 = 1.00 | 3 of 4 = 0.75 |
| 0.2 (flag everything) | 2 of 2 = 1.00 | 4 of 4 = 1.00 |

Each row is one confusion matrix. Plotting FPR (x-axis) against TPR (y-axis) and connecting the points gives the ROC curve, a staircase. The area under it is $0.25 \times 0.5 + 0.75 \times 1.0 = $ **0.875**.

**The shortcut gives the same answer.** There are 2 × 4 = 8 (fraud, legit) pairs. Fraud A (0.9) beats all 4 legitimate transactions; fraud C (0.6) beats D, E, and F but not B (0.8). That's 7 of 8 = **0.875**.


### KS (Kolmogorov-Smirnov) table

**What KS means.** Kolmogorov-Smirnov is a statistical test, named after the two mathematicians who developed it, for measuring how different two groups of numbers are. Here the two groups are the scores given to fraud transactions and the scores given to legitimate ones. A good model gives fraud high scores and legitimate transactions low ones, so the two groups look very different.

#### How it works

1. **Sort all transactions by score**, highest first.
2. **Cut them into equal-sized buckets** (10 by default). The top bucket holds the most suspicious transactions.
3. **Keep running totals going down the table:** how much of all fraud has been caught, and how many of all legitimate transactions have been flagged.

Each row answers: "If we reviewed every transaction down to this bucket, how much fraud would we catch, and how many good customers would we bother?" In ROC terms, "fraud caught so far" is the true positive rate and "legit flagged so far" is the false positive rate at that bucket's cutoff.

#### Example

1,000 transactions (50 fraud, 950 legitimate), cut into 5 buckets of 200:

| Bucket | Fraud | Legit | Fraud caught so far (detection rate) | Legit flagged so far | Legit flagged per fraud caught | KS |
|---|---|---|---|---|---|---|
| 1 (highest scores) | 30 | 170 | 30/50 = 60% | 170/950 = 17.9% | 170/30 = 5.7 | 42.1 |
| 2 | 10 | 190 | 40/50 = 80% | 360/950 = 37.9% | 360/40 = 9.0 | 42.1 |
| 3 | 5 | 195 | 45/50 = 90% | 555/950 = 58.4% | 555/45 = 12.3 | 31.6 |
| 4 | 3 | 197 | 48/50 = 96% | 752/950 = 79.2% | 752/48 = 15.7 | 16.8 |
| 5 (lowest scores) | 2 | 198 | 50/50 = 100% | 950/950 = 100% | 950/50 = 19.0 | 0 |

**Reading row 1.** Reviewing the top 20% of scores catches 60% of all fraud, while flagging 17.9% of all legitimate transactions, or 5.7 legitimate transactions for every fraud caught.

**The KS column** is fraud caught so far minus legit flagged so far, in percentage points. Row 1: 60 − 17.9 = 42.1. The largest value in the column is the **KS statistic**: the point where the model separates fraud from legitimate best. 0 means no separation; 100 would be perfect. Here it is 42.1.

**How KS relates to AUC.** Both come from the same cutoffs. AUC is the area under the whole ROC curve; KS is the largest vertical gap between the ROC curve and the diagonal "coin flip" line. AUC summarizes performance across all cutoffs, while KS points to the single best cutoff.

**The advisor's goal in this table.** "Very low TFPR at 50 to 60% detection" means: go down to the row where fraud caught reaches 50 to 60%, and check how many legitimate transactions have been flagged by then. Lower is better.


**Which column is TFPR?** TFPR (transaction false positive rate) is a fraud-industry metric: the number of legitimate transactions flagged for each fraud caught, usually written as a ratio like 10:1. 


**About the demo below.** There is no model yet, so the demo scores validation with the single strongest feature from the signal check, just to show how to read the table. Real KS tables come from the modeling pipeline's predictions.

In [30]:
def single_feature_auc(y, x):
    x = pd.Series(x).astype("float64")
    x = x.fillna(x.min() - 1 if x.notna().any() else 0)
    if x.nunique() < 2:
        return np.nan
    a = roc_auc_score(y, x)
    return max(a, 1 - a)

check = [(fam, c) for fam, cols in new_families.items() for c in cols if c not in cat_cols]
check_cols = [c for _, c in check]
signal = pd.DataFrame({
    "family": [fam for fam, _ in check],
    "train_auc": [single_feature_auc(labels["train"], tree["train"][c]) for c in check_cols],
    "validation_auc": [single_feature_auc(labels["validation"], tree["validation"][c]) for c in check_cols],
}, index=check_cols)
print(signal.sort_values("validation_auc", ascending=False).round(3).to_string())

                                             family  train_auc  validation_auc
te_card2                            target_encoding      0.685           0.728
te_R_emaildomain                    target_encoding      0.660           0.683
raw_missing_count                    missing_counts      0.665           0.678
freq_id_31                                frequency      0.660           0.668
na_block_20                          missing_blocks      0.652           0.667
has_identity                                  basic      0.652           0.667
na_block_08                          missing_blocks      0.650           0.666
na_block_14                          missing_blocks      0.653           0.666
na_block_03                          missing_blocks      0.649           0.665
identity_missing_count               missing_counts      0.645           0.663
na_block_11                          missing_blocks      0.645           0.663
na_block_21                          missing_blocks 

**KS table: why.** A fraud model is used by choosing a cutoff: transactions above it get reviewed or blocked. Each cutoff trades fraud caught against good customers bothered. AUC averages over every possible cutoff, including ones no bank would use; the KS table shows the cutoffs that matter, in business terms. The advisor asked the team to judge models this way: sort transactions by score from highest to lowest, cut them into equal buckets, and read the running totals down the table.

**The two numbers that matter.**

- **Detection rate** (`cum_detection_rate`): the share of all fraud caught so far. This is fraud stopped.
- **TFPR, transaction false positive rate** (`cum_fp_ratio`): legitimate transactions flagged for each fraud caught, written as a ratio like 10:1. This is the cost: declined cards, customer friction, and analyst review time. Despite the name, it is a ratio, not the standard ML false positive rate (`cum_fpr`, the share of all legitimate transactions flagged), which the table also shows.

**The advisor's target.** About 10:1 TFPR at 50 to 60% detection: catch at least half the fraud while flagging no more than about ten good transactions per fraud. (The meeting notes say "10%," most likely a transcription of "10 to 1"; worth confirming with Debasmita.)

### How the table is calculated

1. **Sort** all transactions by score, highest first.
2. **Cut** them into 10 equal buckets by position. Bucket 1 is the top 10% most suspicious, bucket 2 the next 10%, and so on.
3. **Count** fraud and legitimate transactions in each bucket.
4. **Keep running totals.** Each row includes its own bucket **plus every bucket above it**. Reading down to bucket $b$ means "flag the top $b \times 10\%$ of transactions."

For bucket $b$, with sums running over bucket 1 through bucket $b$:

$$\text{detection rate}_b = \frac{\text{fraud in buckets } 1 \ldots b}{\text{all fraud}}$$

$$\text{cum\_fpr}_b = \frac{\text{legit in buckets } 1 \ldots b}{\text{all legit}}$$

$$\text{TFPR}_b = \frac{\text{legit in buckets } 1 \ldots b}{\text{fraud in buckets } 1 \ldots b}$$

$$\text{KS}_b = \text{detection rate}_b - \text{cum\_fpr}_b, \qquad \text{KS statistic} = \max_b \text{KS}_b$$

The KS statistic is the cutoff where the model separates fraud from legitimate best: 0 means no separation, 1 is perfect.

### Worked example: the demo

**About the demo.** The function is meant for the modeling pipeline. The demo scores validation with whichever new feature scored highest in the signal check, only to show how to read the table. In the run this text was written from, before Jennifer's features were added, that was `raw_missing_count`; with them, the demo should switch to `te_card2`, and its numbers will differ from the example below.

**Totals.** Validation has 59,054 transactions: 1,851 fraud and 57,203 legitimate.

**Bucket 1** (top 10%, 5,906 transactions: 649 fraud, 5,257 legit):

- Detection rate: 649 ÷ 1,851 = **35.1%**
- cum_fpr: 5,257 ÷ 57,203 = **9.2%**
- TFPR: 5,257 ÷ 649 = **8.1:1**
- KS: 0.351 − 0.092 = **0.259**

**Bucket 2 row** (buckets 1 and 2 together, top 20%: 649 + 321 = 970 fraud, 5,257 + 5,584 = 10,841 legit):

- Detection rate: 970 ÷ 1,851 = **52.4%**
- cum_fpr: 10,841 ÷ 57,203 = **19.0%**
- TFPR: 10,841 ÷ 970 = **11.2:1**
- KS: 0.524 − 0.190 = **0.335**, the highest in the table, so the KS statistic is 0.335

**In words:** flagging the top 20% of validation transactions by this one feature catches about half of all fraud, at about 11.2 legitimate transactions flagged per fraud caught.

### Reading the demo output

- **Scores are negative** because a *lower* `raw_missing_count` means more fraud (transactions with identity data have fewer blanks and more fraud), so the code flipped the sign to put the most suspicious transactions first.
- **Neighboring buckets share boundary values.** The count is a whole number, so many transactions tie on the same value, and a tied group can be split across two buckets in arbitrary order. A real model's scores have far fewer ties.
- **This is not the team's result.** It is what one column achieves with no model. It sets a **baseline**: a real model should clearly beat about 52% detection at 11.2:1 TFPR. If it doesn't, something is wrong.


In [31]:
def ks_table(y, score, buckets=10):
    d = pd.DataFrame({"y": np.asarray(y), "s": np.asarray(score, dtype="float64")})
    d = d.sort_values("s", ascending=False).reset_index(drop=True)
    d["bucket"] = pd.qcut(np.arange(len(d)), buckets, labels=range(1, buckets + 1))
    t = d.groupby("bucket", observed=True).agg(n=("y", "size"), fraud=("y", "sum"),
                                               min_score=("s", "min"), max_score=("s", "max"))
    t["legit"] = t["n"] - t["fraud"]
    t["cum_detection_rate"] = t["fraud"].cumsum() / t["fraud"].sum()
    t["cum_fpr"] = t["legit"].cumsum() / t["legit"].sum()
    t["cum_fp_ratio"] = t["legit"].cumsum() / t["fraud"].cumsum().replace(0, np.nan)
    t["ks"] = t["cum_detection_rate"] - t["cum_fpr"]
    return t

best = signal["validation_auc"].idxmax()
s = tree["validation"][best].astype("float64")
s = s.fillna(s.min() - 1)
if roc_auc_score(labels["validation"], s) < 0.5:
    s = -s
demo = ks_table(labels["validation"], s)
print(f"Demo score: {best} alone (validation). KS = {demo['ks'].max():.3f}")
print(demo.round(3).to_string())

Demo score: te_card2 alone (validation). KS = 0.328
           n  fraud  min_score  max_score  legit  cum_detection_rate  cum_fpr  cum_fp_ratio     ks
bucket                                                                                            
1       5906    742      0.071      0.358   5164               0.401    0.090         6.960  0.311
2       5905    216      0.040      0.071   5689               0.518    0.190        11.329  0.328
3       5905    154      0.028      0.040   5751               0.601    0.290        14.932  0.310
4       5906    170      0.027      0.028   5736               0.693    0.391        17.426  0.302
5       5905    129      0.023      0.027   5776               0.762    0.492        19.926  0.271
6       5905    167      0.022      0.023   5738               0.853    0.592        21.454  0.261
7       5906     87      0.018      0.022   5819               0.900    0.694        23.828  0.206
8       5905     90      0.012      0.018   5815         

## 14. Save the handoff

Outputs go to `outputs/feature_engineering/<run_name>/`: tree and neural-net tables, labels (no Kaggle labels exist), metadata with time, product, and both account keys for slicing results, the config, and every fitted rule.

In [32]:
OUT.mkdir(parents=True, exist_ok=True)
for n in tree:
    tree[n].to_parquet(OUT / f"X_tree_{n}.parquet")
    meta[n].to_parquet(OUT / f"metadata_{n}.parquet")
    if CONFIG["save_nn_track"]:
        nn[n].to_parquet(OUT / f"X_nn_{n}.parquet")
    if n in labels:
        labels[n].to_frame().to_parquet(OUT / f"y_{n}.parquet")

manifest = {
    "config": CONFIG,
    "split_rows": {n: len(f) for n, f in raw.items()},
    "versions": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__},
    "low_card_columns": low_card,
    "high_card_columns": high_card,
    "feature_sets": feature_sets,
    "target_encoding_stability": te_report.reset_index().replace({np.nan: None}).to_dict(orient="records"),
    "new_feature_families": new_families,
    "categorical_columns": cat_cols,
    "missing_blocks": block_names,
    "v_keep": v_keep,
    "account_keys": {k: parts for k, (parts, _) in KEYS.items()},
    "device_rules": DEVICE_RULES,
    "nn_track": {"log1p_columns": log_cols, "onehot_columns": onehot_cols, "columns": nn_cols}
                if CONFIG["save_nn_track"] else None,
}
rules = {
    "frequency_maps": {c: {str(k): float(v) for k, v in mp.items()} for c, mp in freq_maps.items()},
    "target_encoding": {c: {"prior": r["prior"], "alpha": r["alpha"],
                            "encoding": {str(k): float(v) for k, v in r["encoding"].items()}}
                        for c, r in te_rules.items()},
    "target_encoding_fold_edges": edges,
    "product_medians": {str(k): float(v) for k, v in product_medians.items()},
    "global_amount_median": global_amount_median,
    "amount_frequency_map_cents": {str(k): float(v) for k, v in amount_freq_map.items()},
    "missingness_cluster_columns": rep_cols,
    "missingness_cluster_centers": na_km.cluster_centers_.round(4).tolist(),
    "category_vocabularies": vocab,
}
(OUT / "config.json").write_text(json.dumps({**CONFIG, "jennifer": JCONFIG}, indent=2) + "\n")
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str) + "\n")
(OUT / "rules.json").write_text(json.dumps(rules) + "\n")
signal.to_csv(OUT / "signal_check.csv")

family_of = {c: fam for fam, cols in new_families.items() for c in cols}
feature_dictionary = pd.DataFrame({
    "feature": tree_cols,
    "family": [family_of.get(c, "original") for c in tree_cols],
    "type": ["categorical" if c in cat_cols else "numeric" for c in tree_cols],
    "in_reduced_v_set": [c not in v_drop for c in tree_cols],
})
feature_dictionary.to_csv(OUT / "feature_dictionary.csv", index=False)

restored = pd.read_parquet(OUT / "X_tree_train.parquet")
pd.testing.assert_frame_equal(tree["train"], restored)
del restored
print("Saved to", OUT)

Saved to outputs\feature_engineering\fe_final_v1


## Where each teammate's work ended up

| From | Feature or method | In this notebook |
|---|---|---|
| EDA | Loading and joining the raw data, training period | Section 2 (identical) |
| Eric | 80/10/10 time split with timestamp-safe boundaries | Section 2 |
| Eric | Email normalization, email provider | Section 3 |
| Eric and Jean | Device families (EDA rules plus Eric's iPhone/iPad/Mac rules) | Section 3 |
| Eric and Jean | Frequency encoding | Section 6 |
| Eric | Amount vs. product median, amount frequency, cents features, weekday, email relationship | Section 6b |
| Eric | Account history on card + address (count, average amount, ratio, time gap) | Section 7, Key 1 |
| Eric | Missingness counts | Section 8b |
| Jean | Missingness profile clusters | Section 8b |
| Eric | Category lists with `__MISSING__` and `__UNSEEN__`, leakage checks, saved outputs | Sections 10, 12, 14 |
| Jennifer | Target (mean) encoding with a stability table: `te_card2`, `te_P_emaildomain`, `te_R_emaildomain` | Section 10b |
| Jennifer | New device, email and address flags; 1-hour and 24-hour velocity; amount minus prior average; prior spread, z-score and ratio to prior maximum; repeated amounts | Sections 10c to 10f |
| Susie | 7-day and 30-day velocity windows (planned in her config) | Section 10d (`vel_*_7d`, `vel_*_30d`) |
| Susie | Amount, time, missingness, email, device, frequency and account-history features | Already covered by Eric's versions in Sections 3, 6, 6b, 7 and 8b |
| Susie | Individual missing flags (`addr1_missing`, `D1_missing`, email missing) | Not added: block flags and the neural-net missing flags cover them (Section 8b note) |
| Salvador | `id_14` timezone treated as a category | Section 4 (`freq_id_14`) |
| Salvador | Three-decimal amount flag | Section 6b (`amount_has_subcent_precision`) |
| Salvador | Cyclical hour and weekday, `card1` frequency, one-hot `ProductCD` | Sections 6, 6b, 10, 11 (already covered) |
| Salvador | Dropping V columns with correlation above 0.9 | Section 9 (EDA method, Spearman 0.95 within blocks; the threshold is a setting) |
| Salvador | Device brands | Section 3 (covered by the finer device families) |
| Salvador | `D9 × 24` | Not added; see question 10 below |
| Sal | Second account key, distinct-value history, block flags, reduced-V feature set, neural-net table, signal check, KS table, per-run config | Sections 7 to 14 |



## 15. Open questions

### For Debasmita

1. **TFPR target.** We read TFPR as the transaction false positive ratio: legitimate transactions flagged per fraud caught (`cum_fp_ratio` in the KS table). The meeting notes say "10% TFPR at 50 to 60% detection." Did you mean a **10:1** ratio at 50 to 60% detection?
2. **Operating range.** What share of transactions does a fraud team typically flag or review? This decides how finely the KS table should be cut. Ten buckets of 10% are too coarse if the realistic range is 1 to 5%.
3. **Dollar detection.** Should we also report the share of fraud **dollars** caught, not just the share of fraud transactions? Catching one large fraud may matter more than several small ones.
4. **Mean encoding.** Jennifer's target encoding (Section 10b) uses only past data for training rows and a first-half vs second-half stability check. `addr1` drifts, so it is left out. Does Mastercard use a standard stability check (for example, PSI or rank-ordering checks) before target encoding, and would the final model refit the encodings on train plus validation?

### For the team

5. **Card and address codes.** Numeric with frequency encodings (this notebook) or categories (Eric's)? A good first comparison once the modeling pipeline exists.
6. **Key 2.** Does `card1 | email domain | D1n` over-merge people on shared domains like `gmail.com`? The signal check scored the `k2_` features low across all transactions, but Key 2 was built for the address-missing group. Score it within that group, then compare model results with and without the `k2_` features.
7. **Repeated identity signal.** `has_identity`, `freq_id_31`, and several `na_block_*` flags all carry the same "has identity data" signal. Keep all of them, or keep one when reducing features?
8. **Near-empty block flags.** `na_block_01`, `02`, and `12` are almost never 1. Drop them?
9. **KS buckets for the real model.** Use 20 or 100 buckets instead of 10, so the table shows the cutoffs near the advisor's target precisely.
10. **D9 (from Salvador's notebook).** Salvador suggested `D9 × 24` as an hour of day. On its own that is only a rescaling of `D9`, so it adds nothing for trees (same order) or for the neural net (standardized anyway). A cyclical version (sine and cosine of the `D9` hour) could add information if `D9` reflects the customer's local time while `TransactionDT` does not. Worth testing? `D9` is missing for most transactions.
11. **`te_P_emaildomain`.** Its stability is the weakest of the three encoded columns (rank correlation 0.31 between the two halves of training, vs 0.48 for `card2` and 0.68 for `R_emaildomain`), and its signal is modest (0.589 validation AUC). Keep it, or drop it from `JCONFIG["te_columns"]`?
12. **Overlap in history features (from Jennifer).** `beh_k1_new_email` and `k1_prior_distinct_P_emaildomain` come from the same history and may repeat each other. Check them together in the feature comparison.
13. **Velocity windows (from Jennifer and Susie).** The notebook uses 1 hour, 24 hours, 7 days and 30 days. More windows (for example 10 minutes) are one line in `JCONFIG["windows"]`. Note that Kaggle test starts about a month after the labeled data, so its first transactions see empty windows.
